# Reproducing Bennett, Ouellette, Ramirez et al. (2024) from dandiset 001051

Paper. Bennett C, Ouellette B, Ramirez TK, et al. (2024). SHIELD: Skull-shaped hemispheric
implants enabling large-scale electrophysiology datasets in the mouse brain. *Neuron* 112,
2869-2885. [doi:10.1016/j.neuron.2024.06.015](https://doi.org/10.1016/j.neuron.2024.06.015). The
claims file was written from the version of record (main text, STAR Methods and the supplemental
figure legends) and the paper's code, https://github.com/AllenInstitute/SHIELD_Dynamic_Gating_Analysis
at commit 3c09a0dc.

Dandiset. [DANDI:001051](https://dandiarchive.org/dandiset/001051/0.260922.1827), version
0.260922.1827. The paper cites the dandiset without a version (its data statement gives
DANDI:001051 and the dandiset's page). There are two published versions, 0.260825.2232 and
0.260922.1827; both hold the same 666 assets with the same blobs and sizes, and the later one only
updates metadata ("Updated metadata"). The most recent published version is pinned, as the protocol
says when no version is cited.

What the dandiset holds. 99 sessions of 27 mice (one NWB file per session with spikes, unit
metrics, CCF channel locations, stimulus tables, trials, licks, rewards, running and eye tracking)
and 567 LFP files (not used here). It holds the behaving-mouse recordings of Figures 4, 7, 8 and S4;
it does not hold the surgery records, ISI images, histology, MRI or training histories of Figure 3,
the VGAT-ChR2 and Vglut2-Cre mice of Figure 6, the dual-hemisphere recordings of Figure 5, or
which sessions had retracted or sharpened probes (Figure S4D to S4H).

Outside inputs. The paper's repository at commit 3c09a0dc2972381f06393ddf5c000e16462c7037: the
list of the 87 sessions passing ephys QC (read from DynamicGating_ephys_metrics.ipynb), its units,
probes and session tables (for the link claim), and the VBN units and sessions tables
(VBN_units_table.csv, VBN_sessions.csv; the 5-mm window comparison from DANDI:000713), used by the
`with_inputs` claims.

Runtime. About 20 minutes, almost all of it streaming the 99 session files (about 640 MB of
spike times each) in four processes.

| Claim | Tested here |
|---|---|
| link-code-tables (added link) | tested |
| fig3b-surgical-outcomes, fig3c-isi-through-implant, fig3d-histology-intact, tracks-rare-bleeding | blocked, data missing |
| fig3fh-mri-deformation, fig3i-learning-similar | blocked, data missing |
| fig4b-units-areas-per-session, figs4ab-insertions-per-mouse, scale-mice-sessions-insertions, penetration-success | tested |
| fig4b-trajectory-consistency | blocked, metadata insufficient |
| fig4c-drift-discontinuities-rare | blocked, methods underspecified |
| fig4d-unit-drift, fig4d-qc-metrics-comparable, qc-pass-fraction-similar | tested |
| fig4e-unit-yield-stable, fig4e-vbn-yield-drops, fig4e-firing-rate-stable, fig4ef-rf-fraction-stable | tested |
| fig4f-rf-match-retinotopy | blocked, data missing |
| fig4g-visual-responses-similar, behavior-stable-across-days, figs4i-insertion-suppression | tested |
| figs4de-retraction-reduces-drift, figs4f-retraction-yield-similar, figs4gh-sharpened-probes | blocked, metadata insufficient |
| figs3d-duragel-yield-stable, fig5-bilateral-recordings, fig6bc-cortical-silencing, fig6ef-optotagging-mrn | blocked, data missing |
| fig7a to fig7f, fig7-hippocampus-not-engaged, figs6-replay (10 claims) | tested |
| fig8bc, fig8d (two), fig8e, fig8f (5 claims) | tested |

In [ ]:
import time
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats
from scipy.optimize import curve_fit
from statsmodels.stats.multitest import multipletests

import shield

pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_rows", 300)
warnings.filterwarnings("ignore")
started = time.time()

In [ ]:
# Stream the session files: tables first, then one pass over each session's spikes.
assets = shield.list_assets()
session_assets = assets[~assets["path"].str.contains("_probe-")].reset_index(drop=True)
sessions, units = shield.scan_all(session_assets["asset_id"].tolist())
sessions["path"] = session_assets["path"].values
print(f"tables of {len(sessions)} sessions read in {time.time() - started:.0f} s")
SP = shield.spike_pass_all(sessions, units)
print(f"spike pass done after {time.time() - started:.0f} s")

In [ ]:
# The paper's code: the 87 sessions passing ephys QC, its unit and session tables (pinned commit)
PASS = set(shield.passing_session_ids())
sessions["passing"] = sessions["session_id"].isin(PASS)
sessions["day"] = sessions["session_type"].str.replace("EPHYS_", "").astype(int)
units = units.join(sessions.set_index("session_id")[["subject_id", "session_type", "passing", "genotype"]], on="session_id")
units["qc"] = shield.qc_pass(units)
units["good"] = units["quality"] == "good"
units["in_area"] = ~units["structure_acronym"].isin(shield.NOT_IN_AREA)
code_units = shield.read_code_csv("metadata_tables/units_table.csv")
vbn_units = shield.read_code_csv("metadata_tables/VBN_units_table.csv")
vbn_sessions = shield.read_code_csv("metadata_tables/VBN_sessions.csv")
code_sessions = shield.read_code_csv("metadata_tables/dynamic_gating_session_metadata.csv")
print("code units table:", code_units.shape, "; VBN units:", vbn_units.shape, "; VBN sessions:", vbn_sessions.shape)

## Data check

In [ ]:
meta = shield.version_metadata()
print("Dandiset", shield.DANDISET_ID, "version", shield.DANDISET_VERSION, ":", meta["name"])
print("assets:", len(assets), "; session files:", len(session_assets), "; LFP probe files:", len(assets) - len(session_assets),
      f"; total {assets['size'].sum() / 1e12:.2f} TB")
print("subjects in the archive summary:", meta["assetsSummary"].get("numberOfSubjects"))
print("\nsessions per subject and genotype:")
print(sessions.groupby(["genotype"]).agg(mice=("subject_id", "nunique"), sessions=("session_id", "size")).to_string())
print("\nsex by mouse:", sessions.groupby("subject_id")["sex"].first().value_counts().to_dict())
print("session types:", sessions["session_type"].value_counts().sort_index().to_dict())
print("duplicate session identifiers:", int(sessions["session_id"].duplicated().sum()))
print("sessions in the code's 87-session list found in the dandiset:", int(sessions["passing"].sum()), "of", len(PASS))
print("\nsessions outside the 87:")
print(sessions.loc[~sessions["passing"], ["path", "session_id", "session_type", "subject_id", "n_units"]].to_string(index=False))

In [ ]:
print("units:", len(units), "; quality values:", units["quality"].value_counts().to_dict())
print("isi_label values:", units["isi_label"].value_counts().to_dict())
cols = ["presence_ratio", "amplitude_cutoff", "isi_violations", "snr", "isolation_distance", "max_drift",
        "cumulative_drift", "amplitude", "firing_rate", "waveform_duration", "x", "y", "z"]
print(units[cols].describe(percentiles=[0.01, 0.5, 0.99]).T.round(3).to_string())
print("\nCCF coordinates (x anterior-posterior, y dorsal-ventral, z left-right, micrometres); midline z = 5700.")
inb = units[units["in_area"]]
print("annotated units with z > 5700:", int((inb["z"] > 5700).sum()), "; largest z:", inb["z"].max(),
      "; structures:", inb.loc[inb["z"] > 5700, "structure_acronym"].value_counts().head(8).to_dict())
print("units with coordinate -25 (no position):", int((units["x"] == -25).sum()),
      "; of those annotated in an area:", int(((units["x"] == -25) & units["in_area"]).sum()))

All recordings are in the left hemisphere: no annotated unit lies more than 550 micrometres right
of the midline, and those that do are in midline structures. Every unit is labelled `good` or
`noise`, as the Methods' non-noise rule needs. The subjects' genotypes are Sst-IRES-Cre;Ai32,
Vip-IRES-Cre;Ai32, Ai32 and wild type: the VGAT-ChR2 and Vglut2-IRES-Cre mice of Figure 6 are not in
the dandiset.

In [ ]:
spt = pd.DataFrame([{"session_id": k, "first_stim": v["first_stim"], "behavior_start": v["beh_start"],
                     "behavior_end": v["beh_end"], "last_stim_end": v["last_stim_end"],
                     "images": ",".join(v["image_names"]), "im115_presentations": v["im115_n"]} for k, v in SP.items()])
sessions = sessions.merge(spt, on="session_id")
sessions["behavior_min"] = (sessions["behavior_end"] - sessions["behavior_start"]) / 60
print(sessions[["behavior_min", "first_stim", "last_stim_end", "im115_presentations"]].describe().round(1).to_string())
print("\nimage sets:", sessions["images"].value_counts().to_dict())
print("sessions without eye tracking:", sessions.loc[~sessions["has_eye"], "session_id"].tolist())
print("rewards per session (all / auto / earned):")
print(sessions[["n_rewards", "n_rewards_auto", "n_rewards_earned", "n_hit", "n_trials"]].describe().round(1).to_string())
print("earned rewards equal hit trials in", int((sessions["n_rewards_earned"] == sessions["n_hit"]).sum()), "of", len(sessions), "sessions")

The task has four image stimuli per session (three natural images and im115_r at 0.7 contrast) in
place of the 8 images the Methods describe; im115_r at full contrast is in every session. The active
block lasts about an hour and is followed by the receptive-field mapping (Gabors), flashes and
spontaneous activity in most sessions.

Clocks. The spikes, the stimulus tables, licks and rewards come from one file per session and
are on one clock as the file states; a check that they share a clock is that visual responses are
locked to the image onsets: the Figure 4G PSTHs below (VISp units, im115_r) rise within tens of ms
after onset and not before it, in every recording day, and the receptive-field maps of the Gabor
stimulus are significant for a large fraction of VISp units, which they would not be with a
shifted clock. For eye tracking and running the next cell contrasts conditions: running speed around
the change time of hit trials against miss trials (mice slow down to lick after a detected change),
and pupil area around full-field white against black flashes (the pupil constricts after a white
flash). On a shared clock each contrast is flat before time 0 and turns after it, at a latency
that fits the physiology (a lick response follows the change by about 0.4 s; the mouse pupil light
reflex starts some 0.2 to 0.3 s after a flash).

In [ ]:
tt_run = np.arange(500) * 0.01 - 2; tt_pup = np.arange(200) * 0.01 - 0.5
rows = []
for k, v in SP.items():
    c = v["clock"]
    h, m = c[("running", "hit")], c[("running", "miss")]
    d = (h - np.nanmean(h[(tt_run >= -1) & (tt_run < 0)])) - (m - np.nanmean(m[(tt_run >= -1) & (tt_run < 0)]))
    w, b = c[("pupil", "white")], c[("pupil", "black")]
    dp = (w - np.nanmean(w[tt_pup < 0])) - (b - np.nanmean(b[tt_pup < 0]))
    row = {"session_id": k, "hits": c[("n", "hit")], "misses": c[("n", "miss")], "response_latency_s": c["response_latency"]}
    if np.isfinite(d).any():
        post = (tt_run >= 0) & (tt_run < 1.5); pre = (tt_run >= -1) & (tt_run < 0)
        thr = 3 * np.nanmax(np.abs(d[pre]))
        cross = np.where(post & (d < -thr))[0]
        row.update(run_pre_max=np.nanmax(np.abs(d[pre])), run_post_min=np.nanmin(d[post]),
                   run_min_time_s=tt_run[post][np.nanargmin(d[post])], run_onset_s=tt_run[cross[0]] if len(cross) else np.nan)
    if np.isfinite(dp).any():
        post = tt_pup >= 0; pre = tt_pup < 0
        thr = 3 * np.nanmax(np.abs(dp[pre]))
        cross = np.where(post & (dp < -thr))[0]
        row.update(pupil_pre_max=np.nanmax(np.abs(dp[pre])), pupil_post_min=np.nanmin(dp[post]),
                   pupil_min_time_s=tt_pup[post][np.nanargmin(dp[post])], pupil_onset_s=tt_pup[cross[0]] if len(cross) else np.nan)
    rows.append(row)
ck = pd.DataFrame(rows).set_index("session_id")
print("running (cm/s; hit minus miss, each minus its -1 to 0 s mean) and pupil (area, pixels^2; white minus black flashes):")
print(ck.describe().round(3).to_string())
print("sessions without a running contrast:", ck.index[ck["run_post_min"].isna()].tolist(),
      "; without a pupil contrast:", ck.index[ck["pupil_post_min"].isna()].tolist())
print("sessions where the contrast after 0 is not more than 3 times the largest before 0 -- running:",
      ck.index[ck["run_post_min"].notna() & ck["run_onset_s"].isna()].tolist(),
      "; pupil:", ck.index[ck["pupil_post_min"].notna() & ck["pupil_onset_s"].isna()].tolist())
not_shown = sorted(set(ck.index[ck["run_post_min"].isna() | ck["run_onset_s"].isna()])
                   | set(ck.index[ck["pupil_post_min"].isna() | ck["pupil_onset_s"].isna()]))
print("sessions where alignment is not shown for at least one stream (no contrast, no eye tracking or a weak contrast):",
      len(not_shown), "of", len(ck), not_shown)
print("sessions where the minimum falls in the first 0.1 s -- running:", ck.index[ck["run_min_time_s"] < 0.1].tolist(),
      "; pupil:", ck.index[ck["pupil_min_time_s"] < 0.1].tolist())
fig, ax = plt.subplots(1, 2, figsize=(9, 3))
R_ = np.array([v["clock"][("running", "hit")] - v["clock"][("running", "miss")] for v in SP.values()])
P_ = np.array([v["clock"][("pupil", "white")] - v["clock"][("pupil", "black")] for v in SP.values()])
ax[0].plot(tt_run, np.nanmean(R_, 0)); ax[0].axvline(0, color="k", lw=0.5); ax[0].set_xlabel("s from change"); ax[0].set_ylabel("hit minus miss running (cm/s)")
ax[1].plot(tt_pup, np.nanmean(P_, 0)); ax[1].axvline(0, color="k", lw=0.5); ax[1].set_xlabel("s from flash"); ax[1].set_ylabel("white minus black pupil area")
plt.show()
print(f"plotted ranges: running {np.nanmin(np.nanmean(R_, 0)):.2f} to {np.nanmax(np.nanmean(R_, 0)):.2f}, pupil {np.nanmin(np.nanmean(P_, 0)):.1f} to {np.nanmax(np.nanmean(P_, 0)):.1f}")

The running and pupil streams share the stimulus and trial clock, as far as these contrasts show. Running
slows after hits against misses from a median of 0.225 s after the change (minimum at a median 0.57 s;
the median response latency of hits is 0.479 s), and the pupil constricts after white against black
flashes from a median of 0.35 s (minimum at a median 0.59 s); before time 0 the contrasts are small
(median largest deviation 2.5 cm/s against a median minimum of -35.7 cm/s after; 38.7 against -759
pixels squared). Sessions 1173189336 and 1230732154 have no running contrast (no hit or no miss trials),
1187475832 and 1202438738 no eye tracking. In 5 sessions for running (1182427414, 1182628226, 1178693650,
1191375883, 1226010097) and 3 for the pupil (1173741216, 1231476077, 1244070169) the contrast after 0
does not exceed three times the largest before it, so alignment is not shown there; in 1231476077 the
pupil minimum falls in the first 0.1 s. Taken together, alignment of at least one stream is not
shown in 12 of the 99 sessions (listed above). This is consistent with a shared clock, not proof of it at the
10 ms scale.

## link-code-tables: the paper's session list and unit table against the dandiset

> The 87 sessions the paper's code lists as passing ephys QC are sessions of the dandiset, and the
> code's units table (unit ids, counts, metrics, receptive-field p values) agrees with the
> dandiset's units for those sessions.

This link claim is added by the author (protocol, Author, step 1.4): the session list and the
per-unit tables are outside inputs from the paper's repository, matched to the dandiset by session
and unit identifiers.

In [ ]:
p_units = units[units["passing"]]
cu = code_units[code_units["session"].isin(PASS)]
print("code sessions in the list found as dandiset identifiers:", len(PASS & set(sessions["session_id"])), "of", len(PASS))
cs = code_sessions.set_index("ecephys_session_id")
mm = sessions.set_index("session_id").join(cs[["session_type", "mouse_id", "genotype"]], rsuffix="_code")
print("session type agrees:", int((mm["session_type"] == mm["session_type_code"]).sum()), "of", len(mm),
      "; mouse agrees:", int((mm["subject_id"].astype(int) == mm["mouse_id"]).sum()),
      "; genotype agrees:", int((mm["genotype"] == mm["genotype_code"]).sum()))
n_nwb = p_units.groupby("session_id").size()
n_code = cu.groupby("session").size()
cmp = pd.concat([n_nwb, n_code], axis=1, keys=["dandiset", "code"]).fillna(0).astype(int)
print("\nunits in the 87 sessions: dandiset", int(n_nwb.sum()), "; code", int(n_code.sum()))
print("sessions with the same unit count:", int((cmp["dandiset"] == cmp["code"]).sum()), "of", len(cmp))
print(cmp[cmp["dandiset"] != cmp["code"]].to_string())
ids_missing = cu[~cu["unit_id"].isin(units.index)]
print("code units not in the dandiset:", len(ids_missing), "; by structure:", ids_missing["structure_acronym"].value_counts().head(6).to_dict())
# which probes the code units without a dandiset unit sit on, and whether the session file has units on that probe
if "probe_id" in units.columns:
    d_probes = units.groupby("session_id")["probe_id"].apply(lambda x: set(x.dropna().astype(int)))
    pm = ids_missing.groupby(["session", "probe_id"]).size().rename("code_units").reset_index()
    pm["dandiset_units_on_probe"] = [int(((units["session_id"] == s_) & (units["probe_id"] == p_)).sum()) for s_, p_ in zip(pm["session"], pm["probe_id"])]
    pm["code_units_on_probe"] = [int(((cu["session"] == s_) & (cu["probe_id"] == p_)).sum()) for s_, p_ in zip(pm["session"], pm["probe_id"])]
    print("code units not in the dandiset, by session and the code's probe_id:")
    print(pm.to_string(index=False))
    print("dandiset probe_ids in those sessions:", {int(s_): sorted(d_probes.get(s_, set())) for s_ in pm["session"].unique()})
else:
    print("the session files' electrodes table has no probe_id column")
print("their peak channels among the dandiset's unit peak channels:", int(ids_missing["peak_channel_id"].isin(units["peak_channel_id"]).sum()), "of", len(ids_missing))
print("dandiset units (87 sessions) not in the code table:", int((~p_units.index.isin(cu["unit_id"])).sum()))
j = p_units.join(cu.set_index("unit_id")[["max_drift", "firing_rate", "quality", "presence_ratio", "snr"]], rsuffix="_code", how="inner")
rows = []
for sid, g in j.groupby("session_id"):
    rows.append({"session_id": sid, "units": len(g),
                 "max_drift_differs": float((np.abs(g["max_drift"] - g["max_drift_code"]) > 0.01).mean()),
                 "firing_rate_differs": float((np.abs(g["firing_rate"] - g["firing_rate_code"]) > 0.01).mean()),
                 "quality_differs": float((g["quality"] != g["quality_code"]).mean())})
agree = pd.DataFrame(rows)
bad = agree[(agree[["max_drift_differs", "firing_rate_differs", "quality_differs"]] > 0.01).any(axis=1)]
print("\nsessions where more than 1% of matched units differ in a metric:", len(bad), "of", len(agree))
print(bad.round(3).to_string(index=False))
print("in the other", len(agree) - len(bad), "sessions the largest fraction differing is",
      round(float(agree.drop(bad.index)[["max_drift_differs", "firing_rate_differs", "quality_differs"]].max().max()), 4))
# implied recording duration n_spikes / firing_rate per unit: constant within a session when the rate
# describes the same spikes; with the code's firing_rate it scatters if the code's units are others
jj = j.assign(dur_dandiset=j["n_spikes"] / j["firing_rate"], dur_code=j["n_spikes"] / j["firing_rate_code"])
dd = jj.groupby("session_id").agg(units=("dur_dandiset", "size"), dur_dandiset_median=("dur_dandiset", "median"),
                                  dandiset_cv=("dur_dandiset", lambda x: x.std() / x.mean()),
                                  code_cv=("dur_code", lambda x: x.std() / x.mean()),
                                  code_ratio_iqr=("dur_code", lambda x: float(np.subtract(*np.percentile(x / x.median(), [75, 25])))))
dd["differs"] = dd.index.isin(bad["session_id"])
print("\nspike counts against firing rates (implied duration n_spikes / firing_rate, s):")
print(dd.groupby("differs")[["dandiset_cv", "code_cv", "code_ratio_iqr"]].median().round(4).to_string())
print(dd[dd["differs"]].round(4).to_string())
print("largest across sessions:", dd.groupby("differs")[["dandiset_cv", "code_cv"]].max().round(4).to_dict("index"))


In [ ]:
# receptive-field p values computed here (chi-square bootstrap) against the code's p_value_rf
rf = pd.concat([v["rf"].assign(session_id=k) for k, v in SP.items()])
rfj = rf.join(code_units.set_index("unit_id")[["p_value_rf"]], how="inner").dropna()
ok_s = ~rfj["session_id"].isin(bad["session_id"])
for lab, g in (("sessions whose metrics agree", rfj[ok_s]), ("the sessions that differ", rfj[~ok_s])):
    print(f"{lab}: {len(g)} units; Pearson r of p values {np.corrcoef(g['p_rf'], g['p_value_rf'])[0, 1]:.4f}; "
          f"same significance at 0.01 for {np.mean((g['p_rf'] < 0.01) == (g['p_value_rf'] < 0.01)):.3f}; "
          f"significant here {np.mean(g['p_rf'] < 0.01):.3f}, in the code {np.mean(g['p_value_rf'] < 0.01):.3f}")
# per session, in the sessions whose metrics agree: fraction of units whose significance at 0.01 differs
ps_ = rfj[ok_s].assign(diff=(rfj["p_rf"] < 0.01) != (rfj["p_value_rf"] < 0.01)).groupby("session_id")["diff"].agg(["size", "mean"])
print(f"per session (sessions whose metrics agree, {len(ps_)} sessions with RF units): fraction differing median {ps_['mean'].median():.3f}, "
      f"max {ps_['mean'].max():.3f}; sessions with more than 1% differing: {int((ps_['mean'] > 0.01).sum())}")
print("those sessions (units, fraction differing):", ps_[ps_["mean"] > 0.01].round(3).to_dict("index"))
near = rfj[ok_s]
dif = near[(near["p_rf"] < 0.01) != (near["p_value_rf"] < 0.01)]
print(f"units whose significance differs: {len(dif)}; code p of these from {dif['p_value_rf'].min():.3f} to {dif['p_value_rf'].max():.3f}, "
      f"p here from {dif['p_rf'].min():.3f} to {dif['p_rf'].max():.3f}; with both p values within 0.005 of 0.01: "
      f"{int(((dif['p_rf'] - 0.01).abs() <= 0.005).sum() if len(dif) else 0)} of those with p here near, "
      f"{int((((dif['p_rf'] - 0.01).abs() <= 0.005) & ((dif['p_value_rf'] - 0.01).abs() <= 0.005)).sum())} both")


**Result: partially reproduced.** All 87 listed sessions are dandiset sessions, and session type, mouse
and genotype agree for all 99. In 81 of the 87 sessions the code's unit table and the dandiset agree
unit by unit (no metric differs). In 6 sessions (1176989662, 1178460518, 1178693650, 1179911454,
1180266229, 1181096406) the same unit ids carry other values for 55 to 99.8% of units. There the
files' spike counts fit the files' firing rates (the implied duration n_spikes / firing_rate has a
coefficient of variation of 0 in five sessions and 0.076 in 1176989662) but not the code's (5.4 to 11.8,
against at most 0.0003 in the other 81 sessions), and the receptive-field p values computed here do not correlate
with the code's (r = 0.03 over 251 units, against r = 0.9990 over 7095 units elsewhere): the code's
table describes other units under those ids than the files hold. In 5 of those sessions the code's
table also has 328 units that are not in the dandiset; each session's are on one probe_id of the
code's (38 to 114 units) that is not among the session file's probe_ids and has no unit in the file,
and none of their peak channels is a peak channel of a dandiset unit. Where the units agree, RF significance at 0.01
agrees for 98.8% of units pooled; per session the fraction differing has a median of 0.008 and a
maximum of 0.038, above 1% in 33 of 79 sessions (listed above). All 83 differing units have p values
from 0.003 to 0.020 in both computations, at the threshold, as two bootstraps of 1000 resamplings with
different random draws give.

## scale-mice-sessions-insertions

> Stable recordings were made from 25 mice in 87 experiments with 467 probe insertions.

An insertion is a probe of a session with at least one unit, as the code counts distinct probe ids
in its units table (DynamicGating_ephys_metrics.ipynb, the cell printing number_of_insertions).

In [ ]:
ins = units.groupby(["session_id", "probe"]).size().rename("n").reset_index()
ins = ins.merge(sessions[["session_id", "subject_id", "passing", "session_type"]], on="session_id")
code_probes = shield.read_code_csv("metadata_tables/probes_table.csv")
cpk = code_units[["session", "probe_id"]].drop_duplicates().merge(code_probes[["id", "name"]], left_on="probe_id", right_on="id")
code_keys = set(zip(cpk["session"], cpk["name"]))
ins["in_code"] = [k in code_keys for k in zip(ins["session_id"], ins["probe"])]
for lab, g in (("87 QC-passing sessions", ins[ins["passing"]]), ("all 99 sessions", ins)):
    print(f"{lab}: mice {g['subject_id'].nunique()}, experiments {g['session_id'].nunique()}, insertions {len(g)}; "
          f"of these also in the code's units table {int(g['in_code'].sum())}")
pa_ = ins[ins["passing"] & ~ins["in_code"]]
print(f"insertions of the 87 sessions not in the code's table: {len(pa_)}, with {int(pa_['n'].sum())} units, labelled",
      units.set_index(["session_id", "probe"]).loc[list(zip(pa_["session_id"], pa_["probe"])), "structure_acronym"].value_counts().to_dict())
print("probes in the code's probes table for the 87 sessions:",
      int(shield.read_code_csv("metadata_tables/probes_table.csv").query("session in @PASS").shape[0]))

**Result: reproduced.** Over the 87 sessions: 25 mice, 87 experiments and 507 insertions with units,
within the tolerance of 467 (46.7). Of the 507, 467 are in the code's units table, the paper's number
exactly; the other 40 are probes whose 2004 units are almost all on unannotated tracks ('Track not
annotated' 1873), which the paper's tables leave out.

## penetration-success

> On average 5.46 of the 6 intended probes per session were successfully inserted.

In [ ]:
for lab, g in (("(a) 87 QC-passing sessions", ins[ins["passing"]]), ("(b) all 99 sessions", ins),
               ("(a), insertions in the code's table only", ins[ins["passing"] & ins["in_code"]])):
    per = g.groupby("session_id").size()
    print(f"{lab}: {len(g)} insertions / {per.size} sessions = {len(g) / per.size:.3f} probes per session; "
          f"sessions with 6, 5, 4 or fewer probes: {(per == 6).sum()}, {(per == 5).sum()}, {(per <= 4).sum()}")
for lab, q in (("87 sessions", code_probes[code_probes["session"].isin(PASS)]), ("99 sessions", code_probes)):
    print(f"code's probes table, {lab}: {len(q)} probes in {q['session'].nunique()} sessions, inserted {q['inserted'].value_counts().to_dict()}, "
          f"{len(q) / q['session'].nunique():.3f} per session")


**Result: reproduced.** 5.83 probes per session over the 87 sessions and 5.84 over all 99, within
5.46 +- 0.546; counting only the insertions in the code's table, 5.37. No reading gives 5.46 exactly. The code's probes table lists 467 probes for the 87 sessions (5.368 per session) and 533 for the 99
(5.384), every one marked inserted, so it records no failed attempt and gives no direct penetration count
either.

## fig4b-units-areas-per-session

> A typical 6-probe session records 1,821 non-noise units (SD 341.9) from 28.1 areas (SD 5.1).

In [ ]:
ia = units[units["passing"] & units["in_area"]]
nprobe = ia.groupby("session_id")["probe"].nunique()
six = nprobe[nprobe == 6].index
d6 = ia[ia["session_id"].isin(six)]
areas = d6.groupby("session_id")["structure_acronym"].nunique()
nunits = d6[d6["good"]].groupby("session_id").size()
print(f"6-probe sessions: {len(six)} of {nprobe.size}")
print(f"areas per session: mean {areas.mean():.2f}, SD {areas.std():.2f} (range {areas.min()} to {areas.max()})")
print(f"non-noise units per session: mean {nunits.mean():.1f}, SD {nunits.std():.1f} (range {nunits.min()} to {nunits.max()})")
by_mouse = d6[d6["good"]].groupby(["subject_id", "session_id"]).size().groupby("subject_id").mean()
print("mean non-noise units per 6-probe session, by mouse:", by_mouse.round(0).astype(int).to_dict())
d6c = d6[~d6["session_id"].isin(bad["session_id"])]
print(f"leaving out the 6 sessions whose units differ from the code's: {d6c['session_id'].nunique()} sessions, units mean "
      f"{d6c[d6c['good']].groupby('session_id').size().mean():.1f}, areas mean {d6c.groupby('session_id')['structure_acronym'].nunique().mean():.2f}")

**Result: reproduced.** Over the 44 six-probe sessions, 1821.0 non-noise units (SD 341.9) from 28.11
areas (SD 5.12), the paper's values.

## figs4ab-insertions-per-mouse

> Across recording days each mouse received about 20 probe insertions (SD 4.1) into about 10 implant
> holes (10.4, SD 2.3).

The implant hole of an insertion is not in the dandiset, so only the insertions part is tested.

In [ ]:
for lab, g in (("(a) 87 QC-passing sessions", ins[ins["passing"]]), ("(b) all sessions", ins),
               ("(a), insertions in the code's table only", ins[ins["passing"] & ins["in_code"]])):
    per = g.groupby("subject_id").size()
    print(f"{lab}: {per.size} mice, insertions per mouse mean {per.mean():.2f}, SD {per.std():.2f}, "
          f"range {per.min()} to {per.max()}")
print("sessions per mouse (all sessions):", sessions.groupby("subject_id").size().value_counts().sort_index().to_dict())

**Result: partially reproduced.** Insertions per mouse are 20.28 (SD 3.34) over the 87 sessions and
21.41 over all sessions, within 20 +- 4.1 (18.68 with only the code's insertions). The implant holes
per mouse could not be tested: no file records the hole of an insertion.

## fig4d-unit-drift

> Unit drift with SHIELD is only slightly higher than with the 5-mm window (median 25.1 against
> 22.82 micrometres).

The 5-mm window units are the VBN units table of the paper's repository (from DANDI:000713).

In [ ]:
dg_good = units[units["passing"] & units["good"]]
vbn = vbn_units.merge(vbn_sessions[["ecephys_session_id", "session_number"]], on="ecephys_session_id")
vbn_good = vbn[vbn["quality"] == "good"]
print("non-noise units: SHIELD", len(dg_good), "(paper 143,828); VBN", len(vbn_good), "(paper 262,177)")
def boot_sem_median(x, rng, n=1000):
    return np.std([np.median(rng.choice(x, len(x))) for _ in range(n)])
rng = np.random.default_rng(4)
x, y = dg_good["max_drift"].dropna().values, vbn_good["max_drift"].dropna().values
print(f"median max_drift: SHIELD {np.median(x):.2f} (bootstrap SE {boot_sem_median(x, rng, 200):.3f}), "
      f"VBN {np.median(y):.2f} (SE {boot_sem_median(y, rng, 200):.3f}); difference {np.median(x) - np.median(y):.2f} um")
print(f"two-sided Mann-Whitney p = {stats.mannwhitneyu(x, y).pvalue:.2e}")
xc = dg_good.loc[~dg_good["session_id"].isin(bad["session_id"]), "max_drift"]
in_code_keys = set(zip(ins.loc[ins["in_code"], "session_id"], ins.loc[ins["in_code"], "probe"]))
dg_in = np.array([k in in_code_keys for k in zip(dg_good["session_id"], dg_good["probe"])])
print("non-noise units in the code's units table for the 87 sessions:", int((cu["quality"] == "good").sum()),
      "; SHIELD non-noise units on insertions in the code's table:", int(dg_in.sum()), ", on the 40 others:", int((~dg_in).sum()))
print("non-noise units outside the 6 sessions that differ: dandiset", int((~dg_good["session_id"].isin(bad["session_id"])).sum()),
      ", code", int(((cu["quality"] == "good") & ~cu["session"].isin(bad["session_id"])).sum()))
print(f"SHIELD median without the 6 sessions that differ from the code: {xc.median():.2f}")
print("SHIELD median by mouse:", dg_good.groupby("subject_id")["max_drift"].median().round(1).to_dict())

**Result: reproduced.** Median max_drift is 25.05 um for SHIELD (144,030 non-noise units) and 22.82
for the VBN units (262,177), within 25.1 +- 0.1 and 22.82 +- 0.05. Of the 144,030, 143,828 (the paper's count
exactly) are on the 467 insertions in the code's table and 202 on the 40 insertions it leaves out.
The code's own table has 144,068 non-noise units for the 87 sessions, which differ from the files' in
the 6 sessions where the table describes other units (other quality labels, and 328 units not in the
files); outside those 6 sessions both give 135,334.

## fig4d-qc-metrics-comparable

> The distributions of unit quality metrics (presence ratio, amplitude cutoff, ISI violations, SNR,
> isolation distance, drift, amplitude) are comparable for SHIELD and the 5-mm window.

In [ ]:
metrics = ["presence_ratio", "amplitude_cutoff", "isi_violations", "snr", "isolation_distance", "max_drift", "amplitude"]
rows = []
for m in metrics:
    a, b = dg_good[m].dropna().values, vbn_good[m].dropna().values
    rows.append({"metric": m, "SHIELD_median": np.median(a), "VBN_median": np.median(b),
                 "ratio": np.median(a) / np.median(b), "within_10pct": abs(np.median(a) / np.median(b) - 1) <= 0.10,
                 "mannwhitney_p": stats.mannwhitneyu(a, b).pvalue,
                 "rank_biserial": 1 - 2 * stats.mannwhitneyu(a, b).statistic / (len(a) * len(b))})
qm = pd.DataFrame(rows)
print(qm.to_string(index=False, float_format=lambda v: f"{v:.4g}"))
print("(a) metrics not significant at 0.05:", int((qm["mannwhitney_p"] >= 0.05).sum()), "of 7;",
      " (b) metrics with medians within 10%:", int(qm["within_10pct"].sum()), "of 7")
fig, ax = plt.subplots(1, 7, figsize=(16, 3))
for i, m in enumerate(metrics):
    ax[i].boxplot([dg_good[m].dropna(), vbn_good[m].dropna()], showfliers=False)
    ax[i].set_xticklabels(["SHIELD", "VBN"]); ax[i].set_title(m, fontsize=8)
plt.tight_layout(); plt.show()
print("Boxes without outliers; whiskers at 1.5 IQR.")

**Result: not reproduced.** Under reading (a) all seven metrics differ (Mann-Whitney p from 3.9e-292 to
3.9e-6), as in the code's own output; under reading (b) 5 of 7 medians are within 10%, with
isi_violations (ratio 0.79) and amplitude_cutoff (0.89) lower for SHIELD. The differences are small
(rank-biserial |r| at most 0.07) and the two outside 10% favour SHIELD.

## qc-pass-fraction-similar

> The fraction of units passing the standard quality filter is very similar for SHIELD (41.1%) and
> the 5-mm window (40.7%).

In [ ]:
vq = (vbn_good["isi_violations"] < 0.5) & (vbn_good["amplitude_cutoff"] < 0.1) & (vbn_good["presence_ratio"] > 0.95)
print(f"SHIELD: {dg_good['qc'].sum()} of {len(dg_good)} non-noise units pass = {100 * dg_good['qc'].mean():.2f}%")
print(f"VBN: {vq.sum()} of {len(vbn_good)} = {100 * vq.mean():.2f}%")
print("SHIELD per mouse (%):", (100 * dg_good.groupby("subject_id")["qc"].mean()).round(1).to_dict())
print(f"SHIELD without the 6 differing sessions: {100 * dg_good.loc[~dg_good['session_id'].isin(bad['session_id']), 'qc'].mean():.2f}%")

**Result: reproduced.** 41.09% of SHIELD non-noise units and 40.69% of VBN units pass the filter,
against 41.1% and 40.7%.

## fig4e-unit-yield-stable

> About 300 units are recorded per probe on each day, and unit yield does not differ across the four
> recording days with SHIELD (p = 0.48).

In [ ]:
yc = dg_good.groupby(["subject_id", "probe", "session_type"]).size().rename("units").reset_index()
days = ["EPHYS_1", "EPHYS_2", "EPHYS_3", "EPHYS_4"]
groups = [yc.loc[yc["session_type"] == d, "units"].values for d in days]
for d, g in zip(days, groups):
    print(f"{d}: {len(g)} insertions, median {np.median(g):.1f} non-noise units (IQR {np.percentile(g, 25):.0f} to {np.percentile(g, 75):.0f})")
kw = stats.kruskal(*groups)
print(f"Kruskal-Wallis across the four days: H = {kw.statistic:.3f}, p = {kw.pvalue:.3f}")
dg_code = dg_good[[k in code_keys for k in zip(dg_good["session_id"], dg_good["probe"])]]
yca = dg_code.groupby(["subject_id", "probe", "session_type"]).size().rename("units").reset_index()
ga = [yca.loc[yca["session_type"] == d, "units"].values for d in days]
print("insertions in the code's table only: insertions", [len(g) for g in ga], ", medians", [float(np.median(g)) for g in ga],
      f", Kruskal-Wallis p = {stats.kruskal(*ga).pvalue:.3f}")
print("(paper: 129, 126, 121, 91 insertions; p = 0.48; the code printed medians 314, 305.5, 295, 303)")
ycc = dg_good[~dg_good["session_id"].isin(bad["session_id"])].groupby(["subject_id", "probe", "session_type"]).size().rename("units").reset_index()
kw2 = stats.kruskal(*[ycc.loc[ycc["session_type"] == d, "units"].values for d in days])
print(f"without the 6 differing sessions: p = {kw2.pvalue:.3f}, medians",
      [float(np.median(ycc.loc[ycc['session_type'] == d, 'units'])) for d in days])

**Result: reproduced.** Kruskal-Wallis across days p = 0.333 over all 500 insertions with non-noise units
(medians 308, 302, 287.5, 277), and p = 0.477 over the 467 insertions of the code's table (129, 126,
121 and 91 insertions, medians 314, 305.5, 295, 303, the paper's counts); every median is within
300 +- 30.

## fig4e-vbn-yield-drops

> With the 5-mm window, unit yield decreased on the second recording day (p = 6e-22).

In [ ]:
vy = vbn_good.groupby(["session_number", "ecephys_session_id", "ecephys_probe_id"]).size().rename("units").reset_index()
v1, v2 = [vy.loc[vy["session_number"] == k, "units"].values for k in (1, 2)]
print(f"VBN day 1: {len(v1)} insertions, median {np.median(v1):.1f}; day 2: {len(v2)} insertions, median {np.median(v2):.1f}")
ka = stats.kruskal(v1, v2)
kb = stats.kruskal(v1, v2, *groups)
print(f"(a) Kruskal-Wallis VBN day 1 against day 2: H = {ka.statistic:.2f}, p = {ka.pvalue:.2e}")
print(f"(b) Kruskal-Wallis over the code's six groups (two VBN days, four SHIELD days): H = {kb.statistic:.2f}, p = {kb.pvalue:.2e}")

**Result: reproduced.** VBN yield falls from a median of 321 to 261 units per insertion; Kruskal-Wallis
p = 1.2e-24 for day 1 against day 2 (a) and p = 1.2e-19 over the code's six groups (b).

## fig4e-firing-rate-stable

> The average firing rate of recorded units is similar across the four recording days (p = 0.21).

In [ ]:
fr = dg_good.groupby(["session_type", "subject_id", "session_id"])["firing_rate"].median().rename("fr").reset_index()
fg = [fr.loc[fr["session_type"] == d, "fr"].values for d in days]
for d, g in zip(days, fg):
    print(f"{d}: {len(g)} sessions, median of session medians {np.median(g):.3f} Hz (range {g.min():.2f} to {g.max():.2f})")
k = stats.kruskal(*fg)
print(f"Kruskal-Wallis: H = {k.statistic:.3f}, p = {k.pvalue:.3f} (text 0.21; code and figure 0.26)")

**Result: reproduced.** Kruskal-Wallis p = 0.223 across days over 23, 24, 22 and 18 sessions (text 0.21,
code 0.26).

## fig4ef-rf-fraction-stable

> A consistent fraction of VISp units has significant spatial receptive fields on each of the four
> recording days (p = 0.38).

Receptive fields are computed here from the Gabor presentations (chi-square statistic on the 9 x 9
map with the bootstrap p value of the AllenSDK, which the code's p_value_rf uses; Methods,
Receptive Field Mapping). VISp is the ISI label, units pass QC.

In [ ]:
v1u = units[units["passing"] & units["qc"] & (units["isi_label"] == "VISp")].join(rf[["p_rf"]], how="left")
print("QC VISp units:", len(v1u), "; with an RF p value:", int(v1u["p_rf"].notna().sum()))
v1u = v1u.join(code_units.set_index("unit_id")[["p_value_rf"]], how="left")
res = {}
for thr in (0.01, 0.05):
    for src in ("p_rf", "p_value_rf"):
        frac = v1u.assign(sig=v1u[src] < thr).groupby(["session_type", "subject_id", "session_id"])["sig"].mean().reset_index()
        g = [frac.loc[frac["session_type"] == d, "sig"].values for d in days]
        k = stats.kruskal(*g)
        res[(thr, src)] = k.pvalue
        print(f"threshold {thr}, p values {'computed here' if src == 'p_rf' else 'of the code table'}: sessions per day "
              f"{[len(x) for x in g]}, median fraction {[round(float(np.median(x)), 3) for x in g]}, Kruskal-Wallis p = {k.pvalue:.3f}")

**Result: reproduced.** With the Methods' threshold (p < 0.01) the fraction of QC VISp units with a
significant RF does not differ across days (Kruskal-Wallis p = 0.671; median fractions 0.88 to 0.91 over
12, 11, 12 and 11 sessions); at 0.05, p = 0.368; with the code's own p values, 0.864 and 0.813.

## fig4g-visual-responses-similar

> The population response of VISp units to an image is highly similar across the four recording days.

In [ ]:
ps = pd.concat([v["im115_psth"] for v in SP.values()])
sel = units[units["passing"] & units["qc"] & (units["isi_label"] == "VISp")].join(rf[["p_rf"]], how="inner")
sel = sel[sel["p_rf"] < 0.01]
P = ps.loc[sel.index].values
t_ms = np.arange(2500) - 1000
base = P[:, 990:1000].mean(1, keepdims=True)
resp = (P[:, 1000:1250] - base).mean(1)
peak = np.argmax(P[:, 1000:1250], axis=1)
summ = []
fig, ax = plt.subplots(1, 2, figsize=(10, 3.5))
for d in days:
    m = (sel["session_type"] == d).values
    mean = P[m].mean(0); mb = mean - mean[990:1000].mean(); mn = mb / mb.max()
    ax[0].plot(t_ms, mb, label=d); ax[1].plot(t_ms, mn, label=d)
    summ.append({"day": d, "units": int(m.sum()), "sessions": sel.loc[m, "session_id"].nunique(), "mice": sel.loc[m, "subject_id"].nunique(),
                 "median_response_0_250ms_Hz": np.median(resp[m]), "median_peak_ms": np.median(peak[m]),
                 "population_peak_ms": int(np.argmax(mn[1000:1250]))})
for a in ax: a.set_xlim(-100, 400); a.legend(fontsize=7); a.set_xlabel("ms from im115_r onset")
ax[0].set_ylabel("baseline-subtracted rate (Hz)"); ax[1].set_ylabel("normalised")
plt.show()
summ = pd.DataFrame(summ); print(summ.round(2).to_string(index=False))
ka = stats.kruskal(*[resp[(sel["session_type"] == d).values] for d in days])
kb = stats.kruskal(*[peak[(sel["session_type"] == d).values] for d in days])
print(f"(a) mean baseline-subtracted response 0-250 ms: Kruskal-Wallis p = {ka.pvalue:.3g}")
print(f"(b) time of peak 0-250 ms: Kruskal-Wallis p = {kb.pvalue:.3g}")
day_means = np.array([P[(sel["session_type"] == d).values].mean(0)[1000:1250] for d in days])
print("Pearson r between day population responses (0-250 ms):", np.round(np.corrcoef(day_means), 3).tolist())
pre, post = P[:, 900:1000].mean(1), P[:, 1020:1120].mean(1)
print(f"clock check: VISp rate -100 to 0 ms {np.median(pre):.2f} Hz, 20 to 120 ms {np.median(post):.2f} Hz "
      f"(Wilcoxon p = {stats.wilcoxon(post, pre).pvalue:.1e}); sessions where the post-onset rate is higher:",
      int((sel.assign(d=post - pre).groupby('session_id')['d'].median() > 0).sum()), "of", sel["session_id"].nunique())

**Result: reproduced.** 595, 482, 534 and 446 VISp units with significant RFs (expected 595, 482, 536,
446); the day population responses correlate at r = 0.977 to 0.991 over 0 to 250 ms; neither the
per-unit response (Kruskal-Wallis p = 0.343) nor the time of peak (p = 0.70) differs across days.

## behavior-stable-across-days

> Task performance is stable across the four recording days despite repeated insertions (median
> rewards 59.5, 63.5, 56.5, 60.5; p = 0.96).

In [ ]:
for lab, sset in (("87 QC-passing sessions", sessions[sessions["passing"]]), ("all sessions (variant)", sessions)):
    for col, name in (("n_rewards", "(a) all rewards"), ("n_rewards_earned", "(b) earned rewards")):
        g = [sset.loc[sset["session_type"] == d, col].values for d in days]
        k = stats.kruskal(*g)
        print(f"{lab}, {name}: sessions {[len(x) for x in g]}, medians {[float(np.median(x)) for x in g]}, Kruskal-Wallis p = {k.pvalue:.3f}")

**Result: partially reproduced.** Rewards do not differ across days under any reading (Kruskal-Wallis p
from 0.731 to 0.817). The medians do not all match: over the 87 sessions all rewards give 73, 65, 61
and 53.5 (days 2 and 3 within tolerance), earned rewards 67, 59, 55 and 48 (days 2 and 3); over all
sessions all rewards give 68, 63.5, 60 and 56.5 (days 2 to 4) and earned rewards 62.5, 57.5, 54 and
50.5 (days 1 to 3). Medians fall from day 1 to day 4 in every reading.

## figs4i-insertion-suppression

> Activity is transiently suppressed after probe insertion and recovers with a time constant of 25
> to 30 minutes.

The fit window is not given by the paper (the S4I legend plots the mean change "over time from
experiment start"). The main fit uses the longest window all 87 sessions share, 77.3 min from the
first stimulus; this is an assumption, and it runs past the active block (about 65 min) into the
Gabor and flash epochs. The fit is also shown over windows of 30 to 64 min, which keep to the
active block, and per session over each session's own active block.

In [ ]:
def sat(t, a, b, tau):
    return a - b * np.exp(-t / tau)
curves = {k: v["s4i_counts"] for k, v in SP.items() if k in PASS and len(v["s4i_counts"]) > 600}
L = min(len(c) for c in curves.values())
C = np.array([c[:L] for c in curves.values()], float)
tmin = np.arange(L) / 60
print(f"sessions: {len(C)}; common length {L / 60:.1f} min from the first stimulus")
fits = {}
fig, ax = plt.subplots(figsize=(6, 3))
for lab, norm in (("(a) divided by the session mean", C.mean(1, keepdims=True)),
                  ("(b) divided by the mean of the last 10 min", C[:, -600:].mean(1, keepdims=True))):
    y = (C / norm).mean(0)
    ym = y.reshape(-1, 60)[: L // 60].mean(1) if L % 60 == 0 else y[: (L // 60) * 60].reshape(-1, 60).mean(1)
    tm = np.arange(len(ym)) + 0.5
    popt, pcov = curve_fit(sat, tm, ym, p0=[1, 0.3, 20], maxfev=20000)
    fits[lab] = popt
    ax.plot(tm, ym, label=lab[:3])
    print(f"{lab}: tau = {popt[2]:.1f} min (SE {np.sqrt(pcov[2, 2]):.1f}), a = {popt[0]:.3f}, b = {popt[1]:.3f}; "
          f"rate at minute 1 {ym[0]:.3f}, at minute 30 {ym[29]:.3f}, last minute {ym[-1]:.3f}")
ax.set_xlabel("minutes from first stimulus"); ax.set_ylabel("normalised rate"); ax.legend(); plt.show()
# the same per session, to show the spread
taus = []
for c in C:
    ym = (c / c.mean())[: (L // 60) * 60].reshape(-1, 60).mean(1)
    try:
        taus.append(curve_fit(sat, np.arange(len(ym)) + 0.5, ym, p0=[1, 0.3, 20], maxfev=20000)[0][2])
    except Exception:
        taus.append(np.nan)
taus = np.array(taus)
print(f"per-session fits (a): median tau {np.nanmedian(taus):.1f} min, IQR {np.nanpercentile(taus, 25):.1f} to {np.nanpercentile(taus, 75):.1f}, failed {int(np.isnan(taus).sum())}")
first = sessions.set_index("session_id").loc[list(curves), "first_stim"]
print(f"first stimulus at {first.min():.0f} to {first.max():.0f} s of the recording (median {first.median():.0f})")

# the fit window: the fits above use the common length (77.3 min, which runs past the active block
# into the Gabor and flash epochs); the same fit over shorter windows from the first stimulus,
# including the shortest active block, and each window normalised by its own mean
be = np.array([(SP[k]["beh_end"] - SP[k]["first_stim"]) / 60 for k in curves])
bs = np.array([(SP[k]["beh_start"] - SP[k]["first_stim"]) / 60 for k in curves])
print(f"active block from {bs.min():.1f} to {bs.max():.1f} min after the first stimulus (median {np.median(bs):.1f}), "
      f"ending at {be.min():.1f} to {be.max():.1f} min (median {np.median(be):.1f})")
win_rows = []
for W in (30, 40, 45, 50, 60, int(be.min()), L // 60):
    Cw = C[:, : W * 60]
    for lab, normw in (("a", Cw.mean(1, keepdims=True)), ("b", Cw[:, -600:].mean(1, keepdims=True))):
        ym = (Cw / normw).mean(0).reshape(-1, 60).mean(1)
        tm = np.arange(len(ym)) + 0.5
        try:
            popt, pcov = curve_fit(sat, tm, ym, p0=[1, 0.3, 20], maxfev=20000)
            tau, se = popt[2], np.sqrt(pcov[2, 2])
        except Exception:
            tau, se = np.nan, np.nan
        win_rows.append({"window_min": W, "reading": lab, "tau_min": tau, "SE": se, "within_22.5_33": 22.5 <= tau <= 33})
print(pd.DataFrame(win_rows).to_string(index=False, float_format=lambda v: f"{v:.3g}"))
# per-session fits over the active block of each session (from its first stimulus to its end, at most
# the common length)
pts = []
for k, c in zip(curves, C):
    W = min(int((SP[k]["beh_end"] - SP[k]["first_stim"]) // 60), L // 60)
    ym = (c[: W * 60] / c[: W * 60].mean()).reshape(-1, 60).mean(1)
    try:
        pts.append(curve_fit(sat, np.arange(len(ym)) + 0.5, ym, p0=[1, 0.3, 20], maxfev=20000)[0][2])
    except Exception:
        pts.append(np.nan)
pts = np.array(pts)
print(f"per-session fits over each session's active block (a): median tau {np.nanmedian(pts):.1f} min, "
      f"IQR {np.nanpercentile(pts, 25):.1f} to {np.nanpercentile(pts, 75):.1f}, sessions within 22.5 to 33: "
      f"{int(((pts >= 22.5) & (pts <= 33)).sum())} of {len(pts)}, failed {int(np.isnan(pts).sum())}")


**Result: not reproduced.** Activity is lower early in the session and recovers (normalised rate 0.826
in minute 1 against 0.999 at minute 30 and 1.077 at the end, reading a), but the fitted time constant is
36.0 min (SE 2.3) under reading (a) and 35.4 min (SE 2.3) under (b) over the common 77.3 min, above the
range 22.5 to 33 min; per-session fits give a median of 34.8 min (IQR 23.1 to 52.5). The window is an
assumption, but no window brings tau into the range: it grows as the window shortens, 41.2 to 42.2 min
over the shortest active block (64 min), 45.9 to 47.1 at 60 min, 65.2 to 84.0 at 40 to 50 min, and at
30 min the fit finds no recovery (tau unbounded). Per-session fits over each session's active block give
a median of 39.8 min (IQR 25.7 to 66.8), with 26 of 87 sessions inside 22.5 to 33 min.

## Alpha events: detection and the shared method

Events are detected in every session as the claims file's header describes, under each open
reading: area membership L1 (the Methods: ISI label for cortex, else the CCF label; areas with at
least 20 QC units) and L2 (the code: CCF labels, an area takes every unit whose label contains its
acronym, areas with more than 20 units), and the event rule E1 (the Methods: four successive maxima)
and E2 (the code: three successive maxima in a window with more than three). Random-time coherence
is R1 (both areas at the random time) or R2 (the reference at the event, the other area at the
random time). Session sets are (a) the 87 QC-passing sessions and (b) all 99. Every alpha claim
below is shown under all 16 combinations.

Choices stated here. (1) Events are detected on the smoothed population rate (the header's
exponential filter, inside `shield.find_events`), while the autocorrelogram threshold, the CCGs and
the coherence use the unsmoothed 10 ms rate. The claims file defines the population rate as smoothed
without saying which steps use it; the detail is taken from the code's printed counts that the header
quotes (1653 VISp and 113 MOp qualifying events, code cell 12), which the unsmoothed autocorrelogram
matches (1643 to 1704 and 119 to 122 under E2): the table below also shows the counts with the
threshold on the smoothed autocorrelogram (`qualifying_smoothed_acg`, 3035 to 3426 and 152 to 170
under E2), far above the code's. (2) The
Benjamini-Hochberg family of the pair test holds the brain areas only; the CP-RF and CP-noRF groups of
Figure 8F, the unannotated labels and the ISI label "VIS" are left out of it (shown against the
correction over all pairs below). (3) A claim part whose area has fewer than 25 events is reported as
untested (`absent`), not as not significant. (4) VISl is taken with its ISI label VISlm (the L1 maps
label the lateral area VISlm; the CCF label is VISl); both are shown. The area labels of each reading
are listed below: SSp has no subdivisions in these files, RSP has RSPagl, RSPd and RSPv, ACA has ACAd
and ACAv.

In [ ]:
COMBOS = [(L, E, R, S) for L in ("L1", "L2") for E in ("E1", "E2") for R in ("R1", "R2") for S in ("a", "b")]
SETS = {"a": PASS, "b": set(sessions["session_id"])}

def events_table(ref, L, E, S):
    rows = []
    for sid, v in SP.items():
        if sid not in SETS[S]:
            continue
        e = v["alpha"].get((L, ref, E))
        if e is None or not len(e["events"]):
            continue
        ev = e["events"].copy()
        ev["session_id"] = sid
        rows.append(ev)
    return pd.concat(rows, ignore_index=True) if rows else pd.DataFrame()

det = []
for L in ("L1", "L2"):
    for E in ("E1", "E2"):
        for S in ("a", "b"):
            for ref in shield.REFERENCE_AREAS:
                ev = events_table(ref, L, E, S)
                elig = sum(1 for sid, v in SP.items() if sid in SETS[S] and ref in v["areas"][L])
                q = ev[ev["qual"]] if len(ev) else ev
                det.append({"L": L, "E": E, "set": S, "area": ref, "eligible_sessions": elig,
                            "sessions_with_events": ev["session_id"].nunique() if len(ev) else 0,
                            "events": len(ev), "qualifying": len(q),
                            "qualifying_smoothed_acg": int((ev["auto_power_smooth"] > shield.AUTO_THRESHOLD).sum()) if len(ev) else 0,
                            "sessions_over_5_qualifying": int((q.groupby("session_id").size() > 5).sum()) if len(q) else 0,
                            "qualifying_in_those": int(q.groupby("session_id").size().loc[lambda s: s > 5].sum()) if len(q) else 0})
det = pd.DataFrame(det)
print(det.to_string(index=False))
print("(the code printed 50 sessions with 1653 qualifying VISp events and 14 sessions with 113 MOp events; not values of the paper)")

# area labels under L1 and L2 (87 sessions): sessions with at least the minimum of QC units, median units
from collections import Counter
for L in ("L1", "L2"):
    cnt, nu = Counter(), {}
    for sid, v in SP.items():
        if sid in PASS:
            for a, n in v["areas"][L].items():
                cnt[a] += 1; nu.setdefault(a, []).append(n)
    print(f"\n{L} areas (sessions, median units):", {a: (cnt[a], int(np.median(nu[a]))) for a in sorted(cnt)})


In [ ]:
NOT_FAMILY = ["CP-RF", "CP-noRF", "VIS", *shield.NOT_IN_AREA]

def pair_table(ref, L, E, R, S, min_events=25):
    """The Figure 7F pair test: per pair the per-event band coherence minus random, pooled over
    sessions with more than 5 qualifying events; pairs with at least 25 events; Wilcoxon against
    zero; Benjamini-Hochberg across the reference area's regions. The family holds brain areas
    only: CP-RF and CP-noRF (Figure 8F groups), the unannotated labels and the ISI label "VIS"
    (visual cortex without an area) are tested but left out of it; `p_bh_all` is the correction
    over every pair, as in the first version of this notebook."""
    vals, ccg_sum, ccg_n = {}, {}, {}
    for sid, v in SP.items():
        if sid not in SETS[S]:
            continue
        e = v["alpha"].get((L, ref, E))
        if e is None or "coh" not in e or len(e["coh"]["event"]) <= 5:
            continue
        d = e["coh"]["event"] - e["coh"][R]
        for j, a in enumerate(e["coh_names"]):
            if a == ref:
                continue
            vals.setdefault(a, []).append(d[:, j])
            ccg_sum[a] = ccg_sum.get(a, 0) + e["ccg_mean"][a] * e["ccg_n"]
            ccg_n[a] = ccg_n.get(a, 0) + e["ccg_n"]
    rows = []
    lag_ms = (np.arange(200) - 100) * 10
    for a, lst in vals.items():
        x = np.concatenate(lst)
        x = x[np.isfinite(x)]
        if len(x) < min_events:
            continue
        c = ccg_sum[a] / ccg_n[a]
        rows.append({"area": a, "events": len(x), "sessions": len(lst), "mean_diff": x.mean(),
                     "p": stats.wilcoxon(x).pvalue, "ccg_peak": c.max(), "ccg_peak_lag_ms": lag_ms[np.argmax(c)]})
    t = pd.DataFrame(rows)
    if len(t):
        fam = ~t["area"].isin(NOT_FAMILY)
        t["p_bh_all"] = multipletests(t["p"], method="fdr_bh")[1]
        t["p_bh"] = np.nan
        t.loc[fam, "p_bh"] = multipletests(t.loc[fam, "p"], method="fdr_bh")[1]
        t.loc[~fam, "p_bh"] = t.loc[~fam, "p_bh_all"]
        t["sig_pos"] = (t["p_bh"] < 0.05) & (t["mean_diff"] > 0)
        t = t.sort_values("area").reset_index(drop=True)
    return t

PAIRS = {(ref, L, E, R, S): pair_table(ref, L, E, R, S) for ref in shield.REFERENCE_AREAS for (L, E, R, S) in COMBOS}
print("pairs tested per combination (VISp, MOp):",
      {f"{L}{E}{R}{S}": (len(PAIRS[('VISp', L, E, R, S)]), len(PAIRS[('MOp', L, E, R, S)])) for (L, E, R, S) in COMBOS})
changed = [(k, x.area) for k, t in PAIRS.items() if len(t) for x in t.itertuples()
           if x.area not in NOT_FAMILY and ((x.p_bh < 0.05) != (x.p_bh_all < 0.05))]
print("pairs whose significance at q < 0.05 changes with the family:", len(changed), "of",
      sum(int((~t["area"].isin(NOT_FAMILY)).sum()) for t in PAIRS.values() if len(t)), changed)


## fig7a-mop-higher-frequency

> VISp alpha events are rhythmic at 3 to 5 Hz, and MOp events occur in a slightly higher band, 5 to 7 Hz.

Frequency readings: (a) the inverse of the median interval between the maxima detected in the
event's window; (b) the peak of the autocorrelogram's spectrum between 1 and 20 Hz (the limits are a
standard choice that excludes the DC term).

In [ ]:
rows = []
for L in ("L1", "L2"):
    for E in ("E1", "E2"):
        for S in ("a", "b"):
            ev_v = events_table("VISp", L, E, S); ev_m = events_table("MOp", L, E, S)
            ev_v, ev_m = ev_v[ev_v["qual"]], ev_m[ev_m["qual"]]
            for fcol, lab in (("freq_peaks", "a"), ("freq_auto", "b"), ("freq_auto_pad", "b, zero-padded")):
                xv, xm = ev_v[fcol].dropna(), ev_m[fcol].dropna()
                p = stats.ranksums(xm, xv, alternative="greater").pvalue if len(xm) and len(xv) else np.nan
                rows.append({"L": L, "E": E, "set": S, "reading": lab, "VISp_n": len(xv), "VISp_median_Hz": xv.median(),
                             "MOp_n": len(xm), "MOp_median_Hz": xm.median(), "p_MOp_higher": p,
                             "holds": (2.7 <= xv.median() <= 5.5) and (4.5 <= xm.median() <= 7.7) and p < 0.05})
fq = pd.DataFrame(rows)
print(fq.to_string(index=False, float_format=lambda v: f"{v:.3g}"))
for lab, g in fq.groupby("reading"):
    print(f"reading ({lab}): holds in {int(g['holds'].sum())} of {len(g)}; MOp median {g['MOp_median_Hz'].min():.3g} to {g['MOp_median_Hz'].max():.3g} Hz, "
          f"VISp median {g['VISp_median_Hz'].min():.3g} to {g['VISp_median_Hz'].max():.3g} Hz, p {g['p_MOp_higher'].min():.2g} to {g['p_MOp_higher'].max():.2g}")
ev_m = events_table("MOp", "L2", "E2", "a"); ev_m = ev_m[ev_m["qual"]]
print("L2 E2 a, MOp events: share of freq_auto values at exactly 4.5 Hz", round(float((ev_m["freq_auto"] == 4.5).mean()), 3),
      "; values on the 0.5 Hz grid:", ev_m["freq_auto"].value_counts().sort_index().to_dict())
# the detector's band: synthetic sinusoidal rates of 2 to 8 Hz, 20 s each, from 0 to 20 spikes/s (the
# minima must reach the height rule, -1 on the inverted rate)
syn = []
for fz in np.arange(2, 8.01, 0.5):
    tt_ = np.arange(2000) * shield.BIN
    for E in ("E1", "E2"):
        syn.append({"Hz": fz, "rule": E, "events": len(shield.find_events((10 * (1 + np.sin(2 * np.pi * fz * tt_))).astype(np.float32), E)[0])})
syn = pd.DataFrame(syn).pivot(index="Hz", columns="rule", values="events")
print("events found in 20 s of a pure sinusoidal rate, by frequency:", syn.T.to_dict("index"))

**Result: reproduced.** Under frequency reading (b), the autocorrelogram peak, VISp events are at a
median 4.0 Hz and MOp events at 4.5 Hz, higher by the one-sided rank-sum test (p from 5.3e-7 to 0.0016) in
all 8 combinations of L, E and session set. On the 0.5 Hz grid of the 2 s autocorrelogram the MOp median
sits on the lower edge of the widened range (4.5 to 7.7 Hz) and 22.7% of MOp values (L2 E2 a) are exactly
4.5 Hz; with the spectrum zero-padded to 0.049 Hz steps the medians are 4.05 to 4.10 Hz (VISp) and 4.64
to 4.74 Hz (MOp), inside the ranges, with MOp higher at p from 7.1e-7 to 0.0038 in all 8. Under reading
(a), the inverse of the median interval between maxima, VISp is at 4.17 to 4.21 Hz and MOp at 4.35 to
4.55 Hz, and MOp is not significantly higher (p 0.078 to 0.76). The criterion accepts a match under a
reading. The detector itself finds sinusoidal rates of 3 to 5 Hz only (none at 2.5 Hz or below, none at
5.5 Hz or above), so the MOp frequencies are bounded by the event rule and a 5 to 7 Hz MOp band could not
appear under either reading.

## fig7b-low-arousal

> VISp and MOp alpha events occur when the mouse is immobile and the pupil constricted, with pupil
> diameter about 25% smaller than at random session times.

Pupil is the pupil-fit area with blinks removed; the diameter reading takes the square root of the
mean area in each window (an approximation stated here: the square root is taken of window means,
not of samples).

In [ ]:
rows = []
for L in ("L1", "L2"):
    for E in ("E1", "E2"):
        for S in ("a", "b"):
            per = {}
            for ref in shield.REFERENCE_AREAS:
                ev = events_table(ref, L, E, S)
                ev = ev[ev["qual"]].copy()
                ref_mean = ev.groupby("session_id")["pupil_win_mean_rand"].transform("mean")
                ev["chg_area"] = (ev["pupil"] - ref_mean) / ref_mean
                ev["chg_diam"] = (np.sqrt(ev["pupil"]) - np.sqrt(ref_mean)) / np.sqrt(ref_mean)
                per[ref] = ev
                run_e, run_r = ev["running"].dropna(), ev["running_rand"].dropna()
                for meas in ("area", "diam"):
                    c = ev["chg_" + meas].dropna()
                    rows.append({"L": L, "E": E, "set": S, "events": ref, "measure": meas, "n": len(c),
                                 "mean_pct": 100 * c.mean(), "median_pct": 100 * c.median(),
                                 "p_pupil": stats.wilcoxon(c).pvalue if len(c) > 5 else np.nan,
                                 "running_event": run_e.mean(), "running_random": run_r.mean(),
                                 "p_running_lower": stats.ranksums(run_e, run_r, alternative="less").pvalue if len(run_e) > 5 else np.nan})
            both = pd.concat(per.values())
            for meas in ("area", "diam"):
                c = both["chg_" + meas].dropna()
                rows.append({"L": L, "E": E, "set": S, "events": "pooled", "measure": meas, "n": len(c),
                             "mean_pct": 100 * c.mean(), "median_pct": 100 * c.median(),
                             "p_pupil": stats.wilcoxon(c).pvalue if len(c) > 5 else np.nan})
ar = pd.DataFrame(rows)
ar["within_tol_mean"] = (ar["mean_pct"] >= -27.5) & (ar["mean_pct"] <= -22.5)
ar["within_tol_median"] = (ar["median_pct"] >= -27.5) & (ar["median_pct"] <= -22.5)
print(ar.to_string(index=False, float_format=lambda v: f"{v:.3g}"))

**Result: partially reproduced.** At qualifying VISp and MOp events the pupil is smaller (Wilcoxon p at
most 9.1e-15 in every combination) and running is lower (0.03 to 1.16 cm/s at events against 4.1 to 7.7
at random times, one-sided rank-sum p at most 8.0e-6). The size of the change does not match 25%: VISp
events give -12.9 to -15.4% (area, mean) and -7.5 to -8.8% (diameter); MOp events -30.8 to -36.5%
(area) and -17.3 to -20.9% (diameter); pooled -14.0 to -16.3% (area). No reading is within -22.5 to
-27.5%. The pupil and running clocks are checked against the
stimulus and trial clock in the data check (alignment of at least one stream is not shown in 12 of 99
sessions: 2 without a running contrast, 2 without eye tracking, 5 with a weak running and 3 with a
weak pupil contrast).

## fig7b-visp-offset-locked

> VISp alpha events are often triggered by the offset of visual stimuli, while the timing of MOp
> events is not related to visual stimuli.

The stimulus before an event is the last active-behavior image presentation starting before it, as
the code's stimulus-trials table holds the active presentations (its columns join each presentation
to a trial).

In [ ]:
rows = []
for L in ("L1", "L2"):
    for E in ("E1", "E2"):
        for S in ("a", "b"):
            for ref in shield.REFERENCE_AREAS:
                ev = events_table(ref, L, E, S)
                x = ev.loc[ev["qual"], "since_offset"].dropna()
                x = x[(x >= 0) & (x < 0.5)].values
                h = np.histogram(x, np.arange(0, 0.5001, 0.05))[0]
                rows.append({"L": L, "E": E, "set": S, "area": ref, "n_in_0_500ms": len(x), "bins": h.tolist(),
                             "largest_bin_ms": int(np.argmax(h) * 50) if len(x) else -1,
                             "p_chi2": stats.chisquare(h).pvalue if len(x) else np.nan,
                             "p_ks": stats.kstest(x, stats.uniform(0, 0.5).cdf).pvalue if len(x) else np.nan})
off = pd.DataFrame(rows)
print(off.to_string(index=False, float_format=lambda v: f"{v:.3g}"))

**Result: reproduced.** In every combination VISp events cluster 100 to 150 ms after stimulus offset
(the largest bin; KS p 9.2e-233 or less, chi-square p printed as 0), while MOp events do not differ from
uniform (chi-square p 0.41 to 0.53, KS p 0.28 to 0.62).

## figs6-replay

> Activity during post-stimulus VISp alpha events is weakly but significantly correlated with
> activity during the preceding stimulus (r = 0.77, p = 1.8e-6), as if events replay recent
> stimulation.

Per VISp unit the mean rate evoked by each of the session's four image stimuli and the mean rate
during the qualifying events that follow each one; each unit's four values divided by their maximum
(a standard choice for "rows normalised"); per session the average over units; Pearson correlation
over sessions, four points per session. Sessions enter when each of the four images precedes at
least one qualifying event.
The criterion assigns each event to the last presentation before it (code cell 12), which also takes
events that start during an image (time since offset below 0). The S6 legend and the Results speak
of post-stimulus events ("alpha burst events following stimulus presentation", "post-stimulus alpha
events"), so a variant keeps only events 0 to 0.5 s after the offset (`post_offset_only`).
Events that follow an omitted flash are assigned to `omitted`, which is not one of the four images,
and drop out.

In [ ]:
rows = []
for L in ("L1", "L2"):
    for E in ("E1", "E2"):
        for S in ("a", "b"):
            for ew in (0.25, 0.5):
                for vw, post in ((1.0, False), (1.25, False), (1.0, True), (1.25, True)):
                    xs, ys, ux, uy, nsess, nev = [], [], [], [], 0, 0
                    for sid, v in SP.items():
                        if sid not in SETS[S]:
                            continue
                        e = v["alpha"].get((L, "VISp", E))
                        if e is None or "replay" not in e:
                            continue
                        rp = e["replay"]
                        prev = np.array(rp["event_prev_image"], dtype=object)
                        so = rp["event_since_offset"]
                        keep = (so >= 0) & (so < 0.5) if post else np.ones(len(prev), bool)
                        imgs = rp["images"]
                        if not all(((prev == im) & keep).any() for im in imgs):
                            continue
                        evk = rp["evoked"][ew]
                        evr = np.stack([rp["event_rates"][vw][:, (prev == im) & keep].mean(1) for im in imgs], 1)
                        with np.errstate(all="ignore"):
                            a = evk / evk.max(1, keepdims=True)
                            b = evr / evr.max(1, keepdims=True)
                        ok = np.isfinite(a).all(1) & np.isfinite(b).all(1)
                        if ok.sum() == 0:
                            continue
                        xs += list(a[ok].mean(0)); ys += list(b[ok].mean(0)); ux += list(a[ok].ravel()); uy += list(b[ok].ravel())
                        nsess += 1; nev += int(sum(((prev == im) & keep).sum() for im in imgs))
                    r, p = stats.pearsonr(xs, ys) if len(xs) > 3 else (np.nan, np.nan)
                    ru, pu = stats.pearsonr(ux, uy) if len(ux) > 3 else (np.nan, np.nan)
                    rows.append({"L": L, "E": E, "set": S, "evoked_s": ew, "event_s": vw, "post_offset_only": post, "sessions": nsess, "events": nev, "points": len(xs),
                                 "r": r, "p": p, "per_unit_r": ru, "per_unit_points": len(ux), "per_unit_p": pu})
rep = pd.DataFrame(rows)
print(rep.to_string(index=False, float_format=lambda v: f"{v:.3g}"))
rep["holds"] = (rep["r"] >= 0.693) & (rep["r"] <= 0.847) & (rep["p"] < 0.05)
for post, g in rep.groupby("post_offset_only"):
    print(f"{'events 0 to 0.5 s after the offset only' if post else 'all events'}: r {g['r'].min():.3f} to {g['r'].max():.3f}; "
          f"p < 0.05 in {int((g['p'] < 0.05).sum())} of {len(g)}; within 0.77 +- 0.077 with p < 0.05 in {int(g['holds'].sum())}")
so_all = np.concatenate([v["alpha"][("L2", "VISp", "E2")]["replay"]["event_since_offset"] for v in SP.values()
                         if "replay" in v["alpha"].get(("L2", "VISp", "E2"), {})])
print("L2 E2 qualifying VISp events by time since the last active presentation's offset (s):",
      dict(zip(["< -0.25", "-0.25 to 0 (during the image)", "0 to 0.5", "0.5 to 5", "over 5 (after the active block)"],
               np.histogram(so_all, [-np.inf, -0.25, 0, 0.5, 5, np.inf])[0].tolist())))

**Result: not reproduced.** The correlation across sessions of evoked and event responses is r = 0.21
to 0.63 over the 32 combinations (7 or 8 sessions, 28 or 32 points; 7 sessions, as in the paper, under
L1), below 0.693 in all of them; it is significant (p < 0.05) in 18 of 32. Keeping only events 0 to
0.5 s after the offset (the post-stimulus events of the S6 legend; 1119 of the 1656 L2 E2 events over all 99 sessions, 473
starting during an image and 64 after the active block) gives r = 0.19 to 0.53, significant in 14 of 32,
none within tolerance. The per-unit correlation is r = 0.46 to 0.67 (0.52 to 0.66 with post-offset
events only).

In [ ]:
def lookup(t, names):
    """Rows of a pair table for an area given as one or more labels (subdivisions)."""
    return t[t["area"].isin(names)] if len(t) else t

def area_state(t, names, mode="any"):
    r = lookup(t, names)
    if not len(r):
        return "absent"
    s = r["sig_pos"].all() if mode == "all" else r["sig_pos"].any()
    return "sig" if s else "ns"

def show_pairs(ref, areas):
    out = []
    for (L, E, R, S) in COMBOS:
        t = PAIRS[(ref, L, E, R, S)]
        row = {"combo": f"{L} {E} {R} {S}"}
        for a, names in areas.items():
            r = lookup(t, names)
            row[a] = ("; ".join(f"{x.area} {x.mean_diff:+.3f} q={x.p_bh:.2g}{'*' if x.sig_pos else ''}" for x in r.itertuples())
                      if len(r) else "absent")
        out.append(row)
    return pd.DataFrame(out)

## fig7cd-visp-coupling

> During VISp events, VISp is tightly coupled to LP and VISl (sharp CCG peaks near zero lag and a 3 to
> 7 Hz coherence peak), but not to PO or SSp.

Each cell: the region, the mean of event minus random band coherence, the Benjamini-Hochberg q
value, and `*` when significant and positive. Under L2, PO also takes POST (the code's substring
match).

In [ ]:
A = {"LP": ["LP"], "VISl": ["VISl", "VISlm"], "PO": ["PO"], "SSp": ["SSp"]}
print(show_pairs("VISp", A).to_string(index=False))
rows = []
for (L, E, R, S) in COMBOS:
    t = PAIRS[("VISp", L, E, R, S)]
    st = {a: area_state(t, n) for a, n in A.items()}
    coh_ok = st["LP"] == "sig" and st["VISl"] == "sig" and st["PO"] != "sig" and st["SSp"] != "sig"
    untested = ", ".join(a for a in A if st[a] == "absent") or "none"
    c = {a: lookup(t, n) for a, n in A.items()}
    pk = {a: (c[a]["ccg_peak"].max() if len(c[a]) else np.nan) for a in A}
    lg = {a: (c[a].sort_values("ccg_peak")["ccg_peak_lag_ms"].iloc[-1] if len(c[a]) else np.nan) for a in A}
    for w in (20, 50):
        near = all(abs(lg[a]) <= w for a in ("LP", "VISl") if np.isfinite(lg[a]))
        higher = min(pk["LP"], pk["VISl"]) > np.nanmax([pk["PO"], pk["SSp"]]) if np.isfinite([pk["PO"], pk["SSp"]]).any() else np.nan
        rows.append({"combo": f"{L} {E} {R} {S}", "ccg_window_ms": w, "coherence_part": coh_ok, "untested_parts": untested,
                     **{f"peak_{a}": round(pk[a], 3) for a in A}, **{f"lag_{a}": lg[a] for a in A},
                     "ccg_near_zero": near, "ccg_higher": higher})
v7 = pd.DataFrame(rows)
print(v7.to_string(index=False))
print("combinations where the coherence part holds:", v7.loc[v7["coherence_part"], "combo"].unique().tolist())

**Result: blocked (methods underspecified).** The CCG part holds in every combination (LP and VISl peak at
0 to 20 ms lag, peaks 0.335 to 0.508, above PO and SSp at 0.055 to 0.092). The coherence part holds under
R1 in all 8 combinations (LP and VISl significant and positive; PO and SSp not) but under R2 only with
L2 E1: with L1, SSp is also significant (mean +0.008 to +0.011, q 0.0032 and 2.5e-5), and with L2 E2
SSp is (+0.007, q 0.048 and 0.036; PO +0.011, q 0.051, is just above 0.05). Every part is tested in
every combination. The readings disagree, so by the claims file's rule the claim is methods
underspecified.

## fig7cd-mop-coupling

> During MOp events, MOp is coupled to PO and SSp but not to LP or VISl.

In [ ]:
A2 = {"PO": ["PO"], "SSp": ["SSp"], "LP": ["LP"], "VISl": ["VISl", "VISlm"]}
print(show_pairs("MOp", A2).to_string(index=False))
rows = []
for (L, E, R, S) in COMBOS:
    t = PAIRS[("MOp", L, E, R, S)]
    st = {a: area_state(t, n) for a, n in A2.items()}
    c = {a: lookup(t, n) for a, n in A2.items()}
    pk = {a: (c[a]["ccg_peak"].max() if len(c[a]) else np.nan) for a in A2}
    lg = {a: (c[a].sort_values("ccg_peak")["ccg_peak_lag_ms"].iloc[-1] if len(c[a]) else np.nan) for a in A2}
    rows.append({"combo": f"{L} {E} {R} {S}", **st,
                 "coherence_part": st["PO"] == "sig" and st["SSp"] == "sig" and st["LP"] != "sig" and st["VISl"] != "sig",
                 "untested_parts": ", ".join(a for a in A2 if st[a] == "absent") or "none",
                 "contradicted": st["PO"] == "ns" or st["SSp"] == "ns" or st["LP"] == "sig" or st["VISl"] == "sig",
                 **{f"peak_{a}": round(pk[a], 3) for a in A2}, **{f"lag_{a}": lg[a] for a in A2}})
print(pd.DataFrame(rows).to_string(index=False))
m7 = pd.DataFrame(rows)
print("combinations with every part tested and holding:", m7.loc[m7["coherence_part"] & (m7["untested_parts"] == "none"), "combo"].tolist())
print("combinations where a tested part contradicts the claim:", m7.loc[m7["contradicted"], "combo"].tolist())


**Result: blocked, methods underspecified.** The criterion's "reproduced" needs MOp-PO to be
significantly coherent. Under E1 (8 of the 16 combinations) too few MOp events fall in sessions with PO
(fewer than 25), so PO is untested (`absent`) and `coherence_part` is False there. Under E2, in all 8
combinations, MOp is significantly coherent with PO (+0.15 to +0.20) and SSp (+0.19 to +0.25) and not
with LP, and the CCGs of PO and SSp peak at 0 to 20 ms with heights 0.20 to 0.33 against 0.03 to 0.06 for
LP and VISl. VISl is also untested under L1 and under L2 E1, and the table's `coherence_part` counts an
untested VISl as passing, so every part is tested and holds only under L2 E2 (4 combinations; VISl
+0.005 and +0.027, q 0.89 and 0.34). The readings therefore disagree, which the claims file's header
makes methods underspecified. That no tested part contradicts the claim is not the criterion's test.
This outcome was proposed after verification round 2 and agreed in round 3.

## fig7e-lags-straddle-zero

> In the 3 to 7 Hz band, time lags between VISp or MOp and their associated regions straddle zero;
> neither consistently leads or follows.

In [ ]:
rows = []
for (L, E, R, S) in COMBOS:
    for ref in shield.REFERENCE_AREAS:
        t = PAIRS[(ref, L, E, R, S)]
        lagv = {}
        for sid, v in SP.items():
            if sid not in SETS[S]:
                continue
            e = v["alpha"].get((L, ref, E))
            if e is None or "lags" not in e or len(e["coh"]["event"]) <= 5:
                continue
            for a in e["lags"].columns:
                lagv.setdefault(a, []).append(e["lags"][a].values)
        res = []
        for a in shield.LAG_PAIRS[ref]:
            if a not in lagv or not len(lookup(t, [a])) or not lookup(t, [a])["sig_pos"].any():
                continue
            x = np.concatenate(lagv[a]); x = x[np.isfinite(x)]
            res.append({"pair": f"{ref}-{a}", "n": len(x), "median_lag_ms": 1000 * np.median(x),
                        "fraction_positive": np.mean(x > 0), "p": stats.wilcoxon(x).pvalue})
        if res:
            r = pd.DataFrame(res); r["p_bh"] = multipletests(r["p"], method="fdr_bh")[1]
            for x in r.itertuples():
                rows.append({"combo": f"{L} {E} {R} {S}", **x._asdict()})
lagt = pd.DataFrame(rows).drop(columns="Index")
lagt["not_significant"] = lagt["p_bh"] >= 0.05
print(lagt.to_string(index=False, float_format=lambda v: f"{v:.3g}"))
print("pairs not significant (straddle zero under the test):", int(lagt["not_significant"].sum()), "of", len(lagt))

**Result: not reproduced.** On the code's test (Wilcoxon against zero, Benjamini-Hochberg) the lags
differ from zero for most pairs in every combination: 8 of 102 pair results are not significant (VISp
to VISl under L1 only). Median lags are small (-23 to +14 ms), and the fraction of positive lags ranges
from 0.12 (MOp to PO) to 0.97 (VISp to VISal, L1). Whether the claim means that the median lag does not differ from zero (the criterion) or that the
distribution of lags spans zero is asked of a reader (questions/001051.yaml).

## fig7f-visp-network

> Beyond visual cortex and thalamus, VISp alpha events enhance coherence of VISp with ACA, RSP and CP.

In [ ]:
A3 = {"ACA": ["ACA", "ACAd", "ACAv"], "RSP": ["RSP", "RSPd", "RSPv", "RSPagl"], "CP": ["CP"], "LGd": ["LGd"],
      "LP": ["LP"], "VISl": ["VISl", "VISlm"]}
print(show_pairs("VISp", A3).to_string(index=False))
rows = []
for (L, E, R, S) in COMBOS:
    t = PAIRS[("VISp", L, E, R, S)]
    for mode in ("any", "all"):
        st = {a: area_state(t, n, mode) for a, n in A3.items()}
        rows.append({"combo": f"{L} {E} {R} {S}", "subdivisions": mode, **st, "holds": all(s == "sig" for s in st.values())})
f7v = pd.DataFrame(rows)
print(f7v.to_string(index=False))
print("combinations where all six are significant and positive:", int(f7v["holds"].sum()), "of", len(f7v))
print("\nall VISp pairs, L1 E2 R2 a:")
print(PAIRS[("VISp", "L1", "E2", "R2", "a")].round(4).to_string(index=False))

**Result: blocked (methods underspecified).** ACA, RSP, CP, LGd, LP and VISl are all significant and
positive in 14 of 16 combinations when any subdivision of ACA or RSP counts and in 8 of 16 when all
must (22 of the 32 rows of the table, which lists each combination under both modes). Under L1 E2 R1 CP is not (q = 0.1), and with all subdivisions required RSPagl or RSPd or ACAv
fail under R1. The readings disagree.

## fig7f-mop-network

> MOp alpha events enhance coherence of MOp with SSp, PO, MRN and the deep layers of the superior
> colliculus (SCiw).

In [ ]:
A4 = {"SSp": ["SSp"], "PO": ["PO"], "MRN": ["MRN"], "SCiw": ["SCiw"]}
print(show_pairs("MOp", A4).to_string(index=False))
rows = []
for (L, E, R, S) in COMBOS:
    t = PAIRS[("MOp", L, E, R, S)]
    st = {a: area_state(t, n) for a, n in A4.items()}
    rows.append({"combo": f"{L} {E} {R} {S}", **st, "holds": all(s == "sig" for s in st.values())})
f7m = pd.DataFrame(rows)
print(f7m.to_string(index=False))
print("\nall MOp pairs, L1 E2 R2 a:")
print(PAIRS[("MOp", "L1", "E2", "R2", "a")].round(4).to_string(index=False))

**Result: blocked (methods underspecified).** Under E2 SSp, PO, MRN and SCiw are significant and positive
in 3 of 4 L and R combinations; under L2 E2 R1 MRN is not (q = 0.11); under E1 PO, MRN and SCiw have too
few events to be tested. MRN and SCiw each come from one session (32 events under L1 E2). The readings
disagree.

## fig7-hippocampus-not-engaged

> Neither VISp nor MOp alpha events engage the hippocampus.

Hippocampal formation regions are those under "Hippocampal formation" in the code's CCF structure
tree (analysis_code/ccf_structure_tree_2017.csv).

In [ ]:
tree = shield.read_code_csv("analysis_code/ccf_structure_tree_2017.csv")
hpf_id = tree.loc[tree["acronym"] == "HPF", "id"].iloc[0]
hpf = set(tree.loc[tree["structure_id_path"].astype(str).str.contains(f"/{hpf_id}/"), "acronym"])
rows = []
for (L, E, R, S) in COMBOS:
    for ref in shield.REFERENCE_AREAS:
        t = PAIRS[(ref, L, E, R, S)]
        h = t[t["area"].isin(hpf)] if len(t) else t
        rows.append({"combo": f"{L} {E} {R} {S}", "ref": ref, "hippocampal_pairs": len(h),
                     "significant_positive": ", ".join(h.loc[h["sig_pos"], "area"]) if len(h) else "",
                     "tested": ", ".join(f"{x.area} {x.mean_diff:+.3f} q={x.p_bh:.2g}" for x in h.itertuples())})
hp = pd.DataFrame(rows)
print(hp.to_string(index=False))
print("combinations and references with a significant positive hippocampal pair:", int((hp["significant_positive"] != "").sum()), "of", len(hp))

**Result: not reproduced.** For VISp events at least one hippocampal region is significantly and
positively coherent in every combination (DG in all, +0.012 to +0.032; also CA1, CA3, ProS, SUB or POST
under R2, and ProS under L1 E2 R1 and SUB under L2 E2 R1); for MOp events in 4 of 16 (CA1 and DG under L2 E1 R2; CA1, +0.025 at q = 0.044, under L1 E2
R2). The hippocampal effects are small against those of the visual areas (+0.06 to +0.39).

## fig8bc-cp-rf-location

> CP units with significant spatial receptive fields lie mostly in the medial dorsal striatum, midway
> along the anterior-posterior extent of CP.

QC CP units of the 87 sessions, split by RF significance (p < 0.01, computed here). In the left
hemisphere a larger CCF z is more medial and a smaller y more dorsal.

In [ ]:
cpu = units[units["passing"] & units["qc"] & (units["structure_acronym"] == "CP")].join(rf[["p_rf"]], how="inner")
cpu["rf_sig"] = cpu["p_rf"] < 0.01
r_, n_ = cpu[cpu["rf_sig"]], cpu[~cpu["rf_sig"]]
print("orientation check, median z (um): VISp units", units.loc[units["structure_acronym"] == "VISp", "z"].median(),
      "; SSp", units.loc[units["structure_acronym"] == "SSp", "z"].median(), "; midline structures (LSr, SFO)",
      units.loc[units["structure_acronym"].isin(["LSr", "SFO"]), "z"].median(), "(the midline is at 5700)")
print(f"QC CP units with an RF p value: {len(cpu)} ({cpu['session_id'].nunique()} sessions, {cpu['subject_id'].nunique()} mice); "
      f"CP-RF {len(r_)}, CP-noRF {len(n_)}")
for ax_, lab, alt in (("z", "medial-lateral (z)", "greater"), ("y", "dorsal-ventral (y)", "less")):
    p = stats.ranksums(r_[ax_], n_[ax_], alternative=alt).pvalue
    print(f"{lab}: CP-RF median {r_[ax_].median():.0f}, CP-noRF median {n_[ax_].median():.0f} um; one-sided rank-sum p = {p:.2e}"
          f" (two-sided {stats.ranksums(r_[ax_], n_[ax_]).pvalue:.2e})")
print(f"anterior-posterior (x): CP-RF median {r_['x'].median():.0f} um; CP units sampled from {cpu['x'].min():.0f} to {cpu['x'].max():.0f} "
      f"(median {cpu['x'].median():.0f}); CP-RF fraction by AP tercile:",
      cpu.groupby(pd.qcut(cpu["x"], 3))["rf_sig"].mean().round(3).tolist())
print("CP-RF units by mouse:", r_.groupby("subject_id").size().to_dict())

**Result: partially reproduced.** CP-RF units (688 of 2376 QC CP units) are more dorsal than CP-noRF
units (median y 3150 against 3500 um, p = 1.9e-58) but more lateral, not more medial (median z 4075
against 4350 um; one-sided p = 1.0, two-sided 1.8e-33). The CP-RF median AP coordinate is 5388 um, in
the posterior half of the CP sampled (3700 to 6500 um), where the CP-RF fraction rises from 0.10 to 0.45.

## fig8d-cp-rf-larger-response

> CP units with receptive fields respond more strongly to the task's full-field images than CP units
> without receptive fields.

In [ ]:
f8 = pd.concat([v["fig8d"] for v in SP.values() if "fig8d" in v])
u8 = units.loc[f8.index].join(rf[["p_rf"]], how="left")
# every CP unit of the presence-0.9 set, noise ones included, has an RF p value (shield.py), so the
# variant without a noise filter splits noise CP units by RF as the code does
print("CP units in the five sessions without an RF p value:", int((u8["structure_acronym"].eq("CP") & u8["p_rf"].isna()).sum()))
res8 = {}
for filt in ("qc", "presence 0.9 (code variant)"):
    keep = u8["qc"] if filt == "qc" else pd.Series(True, index=u8.index)
    cp_rf = u8.index[keep & (u8["structure_acronym"] == "CP") & (u8["p_rf"] < 0.01)]
    cp_no = u8.index[keep & (u8["structure_acronym"] == "CP") & ~(u8["p_rf"] < 0.01)]
    v1i = u8.index[keep & (u8["isi_label"] == "VISp")]
    def prep(ix):
        X = f8.loc[ix].values.astype(float)
        X = X - X[:, 200:250].mean()
        return shield.exponential_convolve(X, tau=2, axis=1)
    Xr, Xn, Xv = prep(cp_rf), prep(cp_no), prep(v1i)
    p = np.array([stats.ranksums(a, b).pvalue for a, b in zip(Xr.T, Xn.T)])
    sig = multipletests(p, method="fdr_bh")[0]
    t = np.arange(750) - 250
    win = (t >= 0) & (t < 250)
    higher = Xr.mean(0) > Xn.mean(0)
    res8[filt] = dict(Xr=Xr, Xn=Xn, Xv=Xv, sig=sig, t=t)
    print(f"{filt}: CP-RF {len(cp_rf)}, CP-noRF {len(cp_no)}, VISp {len(v1i)} units; significant bins 0-250 ms: {int((sig & win).sum())}, "
          f"of which CP-RF higher: {int((sig & win & higher).sum())}; significant bins before onset: {int((sig & (t < 0)).sum())}; "
          f"first significant bin {t[sig][0] if sig.any() else None} ms")
    print("   units by session (CP-RF, CP-noRF, VISp):",
          {int(s): (int((u8.loc[cp_rf, 'session_id'] == s).sum()), int((u8.loc[cp_no, 'session_id'] == s).sum()), int((u8.loc[v1i, 'session_id'] == s).sum())) for s in shield.CP_RF_SESSIONS})
R8 = res8["qc"]
fig, ax = plt.subplots(figsize=(6, 3)); ax2 = ax.twinx()
ax.plot(R8["t"], R8["Xr"].mean(0), "r", label="CP-RF"); ax.plot(R8["t"], R8["Xn"].mean(0), "b", label="CP-noRF")
ax2.plot(R8["t"], R8["Xv"].mean(0), "k", label="VISp"); ax.set_xlim(-50, 400); ax.legend(loc="upper left"); ax2.legend(loc="upper right")
ax.set_xlabel("ms from image onset"); ax.set_ylabel("CP rate (Hz)"); ax2.set_ylabel("VISp rate (Hz)"); plt.show()

**Result: partially reproduced.** In the five sessions CP-RF units respond more than CP-noRF units in 53
bins between 0 and 250 ms (first at 55 ms), CP-RF higher in all, none before onset, with the Methods' QC
(the criterion's units). The unit counts there (70, 162 and 218) miss the paper's 80 and 205 for CP-RF and
CP-noRF (tolerance 8 and 20.5) and match its 239 for VISp only within tolerance. With the code's filter
(presence ratio above 0.9, noise units kept and split by their RF p values) the counts are 79, 206 and
239, within one unit of the paper's, with 78 significant bins, all CP-RF higher: the paper's counts come
from the code's filter. The effect holds under both, the expected counts only under the variant. Whether the criterion should take the code's filter is asked of a reader
(questions/001051.yaml).

## fig8d-response-latencies

> CP-RF responses lag VISp by about 20 ms, and CP-noRF responses lag CP-RF by about another 20 ms.

In [ ]:
def latencies(X, t):
    m = X.mean(0); w = (t >= 0) & (t < 250)
    mw, tw = m[w], t[w]
    pk = tw[np.argmax(mw)]
    half = tw[np.argmax(mw >= mw.max() / 2)]
    return pk, half
for filt, R8_ in res8.items():
    lv, lr, ln = (latencies(R8_[k], R8_["t"]) for k in ("Xv", "Xr", "Xn"))
    print(f"{filt}: (a) peak  VISp {lv[0]} ms, CP-RF {lr[0]} ms, CP-noRF {ln[0]} ms -> CP-RF minus VISp {lr[0] - lv[0]}, CP-noRF minus CP-RF {ln[0] - lr[0]}")
    print(f"{' ' * len(filt)}  (b) half  VISp {lv[1]} ms, CP-RF {lr[1]} ms, CP-noRF {ln[1]} ms -> CP-RF minus VISp {lr[1] - lv[1]}, CP-noRF minus CP-RF {ln[1] - lr[1]}")

**Result: blocked, methods underspecified.** With latency as the time the population mean first
reaches half its peak (b), CP-RF lags VISp by 16 ms and CP-noRF lags CP-RF by 20 ms, both within
20 +- 5. With the time of peak (a) the lags are 34 and 9 ms, both outside. The two filters (the Methods'
QC and the code's) give the same values. The paper says only that "the response time course of these
two populations was different: CP-RF lagged behind VISp responses by ... 20 ms, whereas CP-noRF had
slower responses" (Results, Fine-scale alpha rhythm coupling between visual cortex and striatum,
paragraph 3), and it gives no latency definition. That wording does not pick an onset measure over a
peak measure, and the claims file lists both readings without preferring either. The readings give
different outcomes and no basis in the paper picks one, so the claim is methods underspecified. Half-peak
latency had been adopted as an assumption after it was seen to match; this outcome replaces it, as
proposed after verification round 2 and agreed in round 3.

## fig8e-cp-rf-more-modulated

> Triggered on VISp alpha events, CP-RF firing rates are more modulated than CP-noRF rates,
> particularly in the troughs of the oscillation.

Sessions with qualifying VISp events (more than 5) in which CP is an area of the session.

In [ ]:
from scipy.signal import argrelextrema
rows = []
for L in ("L1", "L2"):
    for E in ("E1", "E2"):
        for S in ("a", "b"):
            tr = []
            for sid, v in SP.items():
                e = v["alpha"].get((L, "VISp", E))
                if sid not in SETS[S] or e is None or "trig_cp" not in e or len(e["coh"]["event"]) <= 5 or "CP" not in v["areas"][L]:
                    continue
                if len(e["trig_cp"]):
                    tr.append(e["trig_cp"])
            if not tr:
                continue
            T = shield.exponential_convolve(np.concatenate(tr), tau=1, axis=2)
            tt = np.arange(250) * 0.01 - 0.5
            vis, crf, cno = T[:, 0], T[:, 1], T[:, 2]
            p = np.array([stats.ranksums(a, b).pvalue for a, b in zip(crf.T, cno.T)])
            sig = multipletests(p, method="fdr_bh")[0]
            pw = np.array([stats.wilcoxon(a, b).pvalue for a, b in zip(crf.T, cno.T)])
            sigw = multipletests(pw, method="fdr_bh")[0]
            vm = vis.mean(0)
            w01 = (tt >= 0) & (tt <= 1)
            troughs = [i for i in argrelextrema(vm, np.less, order=5)[0] if w01[i]]
            near_trough = np.zeros(250, bool)
            for i in troughs:
                near_trough[max(i - 2, 0): i + 3] = True
            base_r, base_n = crf[:, tt < 0].mean(), cno[:, tt < 0].mean()
            rng_r = np.ptp(crf.mean(0)[w01]); rng_n = np.ptp(cno.mean(0)[w01])
            rows.append({"L": L, "E": E, "set": S, "events": len(T), "sessions": len(tr), "VISp_troughs_s": [round(tt[i], 2) for i in troughs],
                         "sig_bins_ranksum": int(sig.sum()), "sig_bins_at_troughs": int((sig & near_trough).sum()),
                         "sig_bins_signedrank": int(sigw.sum()), "sig_at_troughs_signedrank": int((sigw & near_trough).sum()),
                         "range_CP_RF": rng_r, "range_CP_noRF": rng_n, "range_norm_CP_RF": rng_r / base_r, "range_norm_CP_noRF": rng_n / base_n})
f8e = pd.DataFrame(rows)
f8e["holds_a"] = (f8e["sig_bins_at_troughs"] > 0) & (f8e["range_CP_RF"] > f8e["range_CP_noRF"])
f8e["holds_b"] = (f8e["sig_bins_at_troughs"] > 0) & (f8e["range_norm_CP_RF"] > f8e["range_norm_CP_noRF"])
print(f8e.to_string(index=False, float_format=lambda v: f"{v:.3g}"))

**Result: reproduced.** In every combination (387 to 493 events in 4 or 5 sessions) CP-RF and CP-noRF
rates differ in 112 to 222 bins, 24 or 25 of them at the troughs of the VISp rate, and the CP-RF range
over 0 to 1 s is larger (1.96 to 2.11 against 1.18 to 1.34 Hz; normalised 0.50 to 0.56 against 0.30 to
0.32).

## fig8f-cp-rf-higher-coherence

> During VISp alpha events, CP-RF units are more coherent with VISp in the 3 to 7 Hz band than
> CP-noRF units (p = 3.8e-14, 439 events).

In [ ]:
rows = []
for (L, E, R, S) in COMBOS:
    dr, dn = [], []
    for sid, v in SP.items():
        e = v["alpha"].get((L, "VISp", E))
        if sid not in SETS[S] or e is None or "coh" not in e or len(e["coh"]["event"]) <= 5 or "CP" not in v["areas"][L]:
            continue
        nm = e["coh_names"]
        if "CP-RF" not in nm or "CP-noRF" not in nm:
            continue
        d = e["coh"]["event"] - e["coh"][R]
        dr.append(d[:, nm.index("CP-RF")]); dn.append(d[:, nm.index("CP-noRF")])
    a, b = np.concatenate(dr), np.concatenate(dn)
    ok = np.isfinite(a) & np.isfinite(b)
    w = stats.wilcoxon(a[ok], b[ok])
    rows.append({"combo": f"{L} {E} {R} {S}", "events": int(ok.sum()), "sessions": len(dr), "CP_RF_mean": a[ok].mean(),
                 "CP_noRF_mean": b[ok].mean(), "p": w.pvalue, "holds": (a[ok].mean() > b[ok].mean()) and w.pvalue < 0.05})
f8f = pd.DataFrame(rows)
print(f8f.to_string(index=False, float_format=lambda v: f"{v:.3g}"))

**Result: blocked (methods underspecified).** Under R2 (the code), CP-RF coherence with VISp exceeds
CP-noRF (mean 0.096 to 0.110 against 0.042 to 0.052, paired Wilcoxon p from 6.5e-11 to 1.2e-14); under R1
(the Methods read plainly) it does not (p 0.36 to 0.75). The readings disagree. The code's readings
(L2, E2) give 439 events, the paper's count.

## Blocked claims: what the files hold

The groups of one session file, and the attributes of its probes, show where the data a blocked
claim needs would be.

In [ ]:
f = shield.open_nwb(sessions["asset_id"].iloc[0])
groups = []
f.visititems(lambda n, o: groups.append(n) if o.attrs.get("neurodata_type") and n.count("/") <= 2 else None)
print("groups with a neurodata type:", [g for g in groups if not g.startswith("specifications")])
print("probe device attributes:", {k: str(v) for k, v in f["general/devices/probeA"].attrs.items()})
print("electrode group attributes:", {k: str(v) for k, v in f["general/extracellular_ephys/probeA"].attrs.items()})
print("session metadata:", {k: str(v) for k, v in f["general/metadata"].attrs.items()})
f.close()
print("\ngenotypes:", sorted(sessions["genotype"].unique()))
print("code session table columns:", list(code_sessions.columns))
print("code probes table columns:", list(code_probes.columns))

## Summary

A failure to reproduce here may be a failure of this attempt, not of the paper: see the reasoning and
the differences from the paper's method.

| Claim | Outcome |
|---|---|
| link-code-tables | partially reproduced (6 of 87 sessions hold other units than the code's table) |
| fig3b-surgical-outcomes | blocked, data missing |
| fig3c-isi-through-implant | blocked, data missing |
| fig3d-histology-intact | blocked, data missing |
| tracks-rare-bleeding | blocked, data missing |
| fig3fh-mri-deformation | blocked, data missing |
| fig3i-learning-similar | blocked, data missing |
| fig4b-units-areas-per-session | reproduced |
| figs4ab-insertions-per-mouse | partially reproduced (holes not testable) |
| scale-mice-sessions-insertions | reproduced |
| penetration-success | reproduced |
| fig4b-trajectory-consistency | blocked, metadata insufficient |
| fig4c-drift-discontinuities-rare | blocked, methods underspecified |
| fig4d-unit-drift | reproduced |
| fig4d-qc-metrics-comparable | not reproduced |
| qc-pass-fraction-similar | reproduced |
| fig4e-unit-yield-stable | reproduced |
| fig4e-vbn-yield-drops | reproduced |
| fig4e-firing-rate-stable | reproduced |
| fig4ef-rf-fraction-stable | reproduced |
| fig4f-rf-match-retinotopy | blocked, data missing |
| fig4g-visual-responses-similar | reproduced |
| behavior-stable-across-days | partially reproduced |
| figs4i-insertion-suppression | not reproduced |
| figs4de-retraction-reduces-drift | blocked, metadata insufficient |
| figs4f-retraction-yield-similar | blocked, metadata insufficient |
| figs4gh-sharpened-probes | blocked, metadata insufficient |
| figs3d-duragel-yield-stable | blocked, data missing |
| fig5-bilateral-recordings | blocked, data missing |
| fig6bc-cortical-silencing | blocked, data missing |
| fig6ef-optotagging-mrn | blocked, data missing |
| fig7a-mop-higher-frequency | reproduced |
| fig7b-low-arousal | partially reproduced |
| fig7b-visp-offset-locked | reproduced |
| figs6-replay | not reproduced |
| fig7cd-visp-coupling | blocked, methods underspecified |
| fig7cd-mop-coupling | blocked, methods underspecified |
| fig7e-lags-straddle-zero | not reproduced |
| fig7f-visp-network | blocked, methods underspecified |
| fig7f-mop-network | blocked, methods underspecified |
| fig7-hippocampus-not-engaged | not reproduced |
| fig8bc-cp-rf-location | partially reproduced |
| fig8d-cp-rf-larger-response | partially reproduced (effect holds; unit counts match only with the code's filter) |
| fig8d-response-latencies | blocked, methods underspecified |
| fig8e-cp-rf-more-modulated | reproduced |
| fig8f-cp-rf-higher-coherence | blocked, methods underspecified |

How the analysis differs from the paper's. Receptive fields are recomputed from the Gabor
presentations with the AllenSDK chi-square bootstrap (1000 resamplings, own seed). Population rates
for windows are cut from one 10 ms grid per session (window starts rounded to the grid). Alpha-event
claims are run under the claims file's readings L1/L2, E1/E2, R1/R2 and two session sets; the code's
batch outputs are not used. Events are detected on the smoothed population rate, and the
autocorrelogram threshold, CCGs and coherence use the unsmoothed rate (taken from the code's printed
event counts, which only the unsmoothed autocorrelogram matches). The Benjamini-Hochberg family of the pair test holds brain areas only. Pupil
diameter is taken as the square root of window-mean area. The S4I fit uses 1-minute means and an
exponential recovery a - b exp(-t / tau), fitted by least squares over 77.3 min (an assumption; shorter
windows give longer time constants). The Figure S6 events are assigned to the last presentation before
them, as the criterion says; a variant keeps post-offset events only.

What was found about the dataset. (1) In 6 of the 87 QC-passing sessions the units under each id
differ from the paper's units table (metrics differ for 55 to 99.8% of units; RF p values
uncorrelated; the files' spike counts fit the files' firing rates but not the table's), and in 5 of them 328 units of the paper's table sit on one probe per session that the files hold
no units for. (2) 40
probe recordings in the 87 sessions (2004 units, nearly all on unannotated tracks) are not in the
paper's tables; nothing in the files marks them. (3) No file records which probes were sharpened or
retracted, the implant hole or trajectory of an insertion, or failed insertion attempts. (4) The
dandiset has none of the Figure 3 material, the Figure 5 dual-hemisphere sessions or the Figure 6
VGAT-ChR2 and Vglut2-Cre mice. (5) Two sessions (1187475832, 1202438738) have no eye tracking. (6) The
task uses four image stimuli per session, where the Methods describe 8 natural images.
